# STEMI — Huấn luyện lại local (máy có GPU) — model đơn + ensemble

**Dự án ACS-ECG-AI (Vinmec).** Bản **local-only** của `notebooks/stemi_pipeline_optimized_v3.ipynb`
— dùng để **train lại** các checkpoint đã mất trên Google Drive (tài khoản Colab dùng để train
bị xoá). Giữ nguyên dữ liệu, kiến trúc, logic ensemble/ngưỡng/leakage-control của bản gốc —
chỉ bỏ toàn bộ phần mount Google Drive / Colab. Không cần Colab, không cần tài khoản Google nào
— chạy và lưu hoàn toàn cục bộ trên `datasets/` và `outputs/` nằm cùng cấp với notebook này
trong repo. **Thiết kế cho máy có GPU CUDA (đã kiểm chứng: RTX 2000 Ada).**

Bài toán: phân loại nhị phân **STEMI** (ST-Elevation Myocardial Infarction) từ tín hiệu ECG 12
chuyển đạo, 10 giây @ 500Hz, dùng cho triage cấp cứu tim mạch — ưu tiên **Sensitivity/NPV**
(rule-out) hơn Specificity/PPV vì chi phí bỏ sót một ca nhồi máu cấp lớn hơn nhiều chi phí một
cảnh báo giả.

**Dữ liệu:** bộ ACS-ECG 2026 (Du et al., 19.955 bản ghi đã xác nhận qua chụp mạch vành), đọc
trực tiếp từ `datasets/ECG_row_data/row_data/` (`.dat`/`.hea`, định dạng WFDB) +
`datasets/CSV/train.csv`.

**Lưu ra `outputs/`:**
- `outputs/cache/` — cache tín hiệu tiền xử lý (`.npy` memmap, **resumable**: ngắt giữa chừng
  thì chạy lại notebook sẽ tiếp tục đúng chỗ dở, không tính lại từ đầu). Notebook **OMI dùng lại
  được** cache này (tiền xử lý không phụ thuộc nhãn đích) — nên chạy notebook STEMI này trước.
- `outputs/models/stemi_stemi_optimized_v3/` — checkpoint từng **model đơn** (10 kiến trúc ×
  5 fold, cộng 10 model FINAL train trên toàn POOL).
- `outputs/results/` — bảng metric TEST, so sánh với báo cáo, CI bootstrap, và xác suất dự đoán
  của **ensemble** đã khoá (`stemi_test_predictions_*.npz`).

**Mốc đối chiếu:** số liệu STEMI đang có trong `reports/Bao_cao_nghien_cuu.docx` mục 2.2.1 —
AUROC 0,9517 | AUPRC 0,7435 | Sens 0,9447 | Spec 0,8231 | PPV 0,3198 | F1 0,4779.
Notebook in thẳng bảng chênh lệch với mốc này ở mục 19. Con số mốc **không** tham gia vào bất
kỳ quyết định nào của pipeline. Do nondeterminism của GPU (`cudnn.benchmark=True`, AMP, xem
cảnh báo ở mục 10 của `HUONG_DAN_COLAB.md`), kết quả chạy lại sẽ **gần** chứ khó **trùng khớp
tuyệt đối** mốc này (dao động AUROC quan sát được trước đây ~0,013–0,015 giữa các lần chạy).

---

## Giữ nguyên từ v2 (đã review, đạt chuẩn)

1. **Chia dữ liệu & chống rò rỉ theo bệnh nhân** — POOL/TEST rồi K-Fold trong POOL, không
   bệnh nhân nào xuất hiện ở hai tập, kiểm tra bằng assert tường minh.
2. **Chuẩn hoá z-score fit riêng theo train của từng fold** — không dùng thống kê toàn cục.
3. **Hàm loss chống mất cân bằng** — Focal Loss + label smoothing, α suy ra từ đúng tỷ lệ
   lớp của từng fold.
4. **Tiền xử lý** — highpass 0,05Hz (chuẩn AHA/ACC/HRS cho ECG chẩn đoán, giữ được mức chênh
   ST tuyệt đối), winsorize ±6mV, QC rà soát chuyển đạo flatline, 5 loại augmentation.
   `PREPROCESS_VERSION` **không đổi** so với v2 nên cache tín hiệu ~1,8GB dùng lại được.
5. **Hiệu chuẩn Platt cross-fit** trước khi ensemble.
6. **Cache/checkpoint fingerprinting** — tránh dùng nhầm cache/checkpoint cũ khi đổi tham số.

---

## Sáu thay đổi của v3, và vì sao

### 1. [LỚN NHẤT] Predictor chính đổi từ FINAL sang **BAGGING** (mục 18)

v2 dự đoán TEST bằng **một** checkpoint train trên 100% POOL, trong khi 5 checkpoint fold đã
được train và lưu sẵn ở mục 13 nhưng chỉ dùng để tính OOF rồi bỏ đi. Đo trên chính bộ dữ liệu
này (notebook 12), trung bình 5 checkpoint fold cho AUPRC **cao hơn +0,03 đến +0,05** so với
checkpoint FINAL ở từng model đơn. Mức lợi này không tốn thêm một giây GPU nào.

Quan trọng hơn: ngưỡng được chốt trên OOF — vốn do các model train trên ~80% POOL sinh ra.
Model FINAL train trên 100% POOL nên tự tin hơn và **lệch thang xác suất**, làm ngưỡng chuyển
sang TEST bị sai. Đây chính là nguyên nhân Sensitivity của v2 tụt còn 0,8894 dù chính sách
đặt ra là ≥ 0,91. Bagging khớp thang với OOF nên ngưỡng chuyển đúng hơn hẳn.

### 2. Vườn ensemble được **chọn trên OOF**, thay cho trung bình đều (mục 16)

v2 lấy trung bình đều cả 8 model, nên ConvNeXtV2 — model hỏng, AUPRC OOF 0,5911 so với
0,64–0,67 của phần còn lại — vẫn được tính đủ 1/8 trọng số và kéo cả ensemble xuống.

v3 dựng 6 cách tổ hợp (Avg-Raw, Avg-Cal, LogitAvg-Cal, Greedy/Caruana, Top-K, Stacking) rồi
chấm điểm **chỉ trên OOF** và khoá quán quân trước khi chạm vào TEST. Các phương án có tham
số (Greedy, Top-K, Stacking) đều được **cross-fit** — không có bước này chúng sẽ tự chấm điểm
trên chính dữ liệu vừa fit và luôn thắng một cách giả tạo. Greedy/Top-K tự loại hoặc hạ trọng
số model kém mà không cần can thiệp tay.

### 3. Sửa ConvNeXtV2: `lr 1e-3 → 3e-4` + **warmup 3 epoch** (mục 2, 13)

Chẩn đoán từ v2: ConvNeXtV2 có `best_epoch` dao động 7→28 giữa các fold (std 7,55 — gấp ~3
lần mọi model khác). Đó là dấu hiệu kinh điển của LR quá cao với mạng LayerNorm+GELU+GRN.
Warmup tuyến tính được áp cho **mọi** model (ổn định gradient những epoch đầu, gần như không
bao giờ hại), `ReduceLROnPlateau` chỉ bắt đầu hoạt động sau giai đoạn warmup.

### 4. Thêm 2 kiến trúc **đa dạng**: TCN1D và CNNTransformer (mục 11)

Trong 8 model của v2 có tới 4 cái (ResNet1D, SEResNet1D, XResNet1D, AiTiAMI) đều là
residual-CNN kernel 5–7 — lỗi tương quan cao nên ensemble gần như không lợi thêm. TCN1D
(convolution giãn nở, trường thu nhận tăng theo cấp số nhân) và CNNTransformer (tự chú ý toàn
cục thay vì tuần tự như BiLSTM) sai ở những ca khác — đó mới là thứ ensemble khai thác được.
Rủi ro thấp vì thay đổi số 2 sẽ tự loại chúng nếu chúng yếu.

### 5. Ngưỡng có **biên an toàn** suy từ dao động giữa các fold (mục 19)

Ngưỡng chốt để đạt Sensitivity = S trên POOL gần như không bao giờ cho đúng S trên TEST:
TEST chỉ có ~217 ca dương nên Sensitivity dao động thuần do cỡ mẫu khoảng ±2 điểm %. v2 dính
đúng bẫy này theo chiều xấu.

Cách sửa, thuần POOL: tại đúng ngưỡng vừa chốt, đo Sensitivity đạt được trong **từng fold**
rồi lấy độ lệch chuẩn giữa các fold (mỗi fold ~245 ca dương, cỡ tương đương TEST). Nhắm cao
hơn mục tiêu z lần độ lệch đó (z = 1,28 → xác suất ~90% đạt mục tiêu thật trên TEST).

### 6. **Bootstrap CI theo bệnh nhân** cho mọi kết luận (mục 16, 20)

v2 không có khoảng tin cậy nào, nên không thể biết một thay đổi là cải thiện thật hay nhiễu.
v3 dùng bootstrap 2.000 lần **lấy mẫu theo bệnh nhân** (giống Phần I của báo cáo) cho: ΔAUPRC
giữa các cách tổ hợp trên OOF, và CI 95% của mọi chỉ số trên TEST.

Ngoài ra `EPOCHS` 30 → 40: ở v2 trung vị `best_epoch` là 23–27 và **hai** model chạm trần 30
(ResNet1D, CNN+BiLSTM) — tức chúng vẫn còn đang cải thiện khi bị cắt.

---

## Kiểm soát rò rỉ

Chia theo **bệnh nhân** ở mọi bước (POOL/TEST, rồi K-Fold trong POOL) — không bệnh nhân nào
xuất hiện ở hai tập. TEST (15%) không được chạm tới cho đến mục 18. Model, cách tổ hợp và
ngưỡng đều được khoá trên POOL trước đó. Nếu trên TEST có model khác điểm cao hơn quán quân,
notebook **in ra ghi chú nhưng không đổi** — chọn theo điểm TEST chính là hình thức rò rỉ mà
toàn bộ thiết kế này đang tránh.

## Chi phí chạy (ước lượng)

10 kiến trúc × 5 fold × tối đa 40 epoch (early stop patience 10) ≈ **4–5 giờ trên Colab T4**
(ước lượng gốc, chưa đo trực tiếp trên RTX 2000 Ada), cộng ~1 giờ cho 10 model FINAL. Toàn bộ
có checkpoint theo từng (model, fold) nên **chạy nhiều phiên được**: tắt máy/notebook rồi mở
lại, Run All từ đầu, phần đã xong tự bỏ qua. Muốn rẻ hơn: bớt tên trong `CANDIDATES` ở mục 2 —
checkpoint các model còn lại vẫn dùng lại được.

## 1. Cài đặt thư viện

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("wfdb") is None:
    print("Đang cài wfdb ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wfdb"], check=True)

import wfdb

print("wfdb", wfdb.__version__)

## 2. Cấu hình

Mọi tham số của pipeline (đường dẫn, tiền xử lý, augmentation, loss, huấn luyện, ngưỡng vận
hành) tập trung ở một chỗ. `EXPERIMENT_TAG`/`PREPROCESS_VERSION` được gắn vào tên checkpoint
và cache: đổi các chuỗi này khi sửa logic tiền xử lý/huấn luyện để tránh vô tình dùng nhầm
cache/checkpoint của một cấu hình khác — hash chỉ theo danh sách bản ghi sẽ không phát hiện
được việc code tiền xử lý đã đổi.

In [ ]:
import os
from pathlib import Path

import torch

# ===================== CHẾ ĐỘ CHẠY =====================
RUN_MODE = "full"        # "debug" = 200 bản ghi / 3 epoch (kiểm tra pipeline nhanh) | "full" = toàn bộ

# ===================== ĐƯỜNG DẪN (chỉ máy cá nhân có GPU — không dùng Colab) ============
# Bản local-only: bỏ toàn bộ nhánh Google Colab/Drive của notebook gốc. Tự dò PROJECT_ROOT
# bằng cách đi lên từ thư mục hiện hành tới khi gặp thư mục chứa "datasets/", phòng trường hợp
# Jupyter không khởi động đúng ngay tại thư mục chứa notebook -- không hardcode ổ đĩa/username.
IS_COLAB = False


def _find_project_root(start: Path) -> Path:
    for cand in [start, *start.parents]:
        if (cand / "datasets").is_dir():
            return cand
    raise FileNotFoundError(
        f"Không tìm thấy thư mục 'datasets/' từ {start} hoặc bất kỳ thư mục cha nào -- "
        "mở notebook này từ trong repo ECG-experiment (nơi có sẵn datasets/ và outputs/)."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
DATA_ROOT = PROJECT_ROOT / "datasets"
WORK_DIR = PROJECT_ROOT / "outputs"
PERSIST_DIR = WORK_DIR

CACHE_DIR = WORK_DIR / "cache"
DRIVE_CACHE_DIR = None

EXPERIMENT_TAG = "stemi_optimized_v3"     # đổi mỗi khi sửa loss/augmentation/kiến trúc
MODEL_DIR = PERSIST_DIR / "models" / f"stemi_{EXPERIMENT_TAG}"
OOF_DIR = PERSIST_DIR / "oof"

# ===================== THAM SỐ TÍN HIỆU =========================
SEED = 42
FS, SIGNAL_LEN, NUM_LEADS = 500, 5000, 12      # 12 chuyển đạo × 10 giây @ 500 Hz
# BP_LOW = 0.05 (không phải 0.5) -- theo khuyến nghị AHA/ACC/HRS cho ECG chẩn đoán.
# Ở 0.5Hz, kể cả filter zero-phase (filtfilt, không lệch pha) vẫn suy hao năng lượng gần DC,
# đúng dải chứa mức chênh lệch ST tuyệt đối -- biến số cốt lõi của bài toán STEMI. 0.5Hz từng
# phổ biến trên máy theo dõi nhịp (không phải máy ECG chẩn đoán) và có thể tạo ST chênh giả.
BP_LOW, BP_HIGH, BP_ORDER = 0.05, 40.0, 3      # bandpass Butterworth (zero-phase qua filtfilt)
WINSORIZE_MV = 6.0                             # biên độ sau lọc vượt ±6mV -> nghi artifact, không phải QRS sinh lý
# GIỮ NGUYÊN chuỗi này so với v2: cache tín hiệu (~1,8GB) được đánh dấu theo nó, đổi là phải
# tiền xử lý lại toàn bộ. v3 không thay đổi gì trong preprocess() nên cache cũ dùng lại được.
PREPROCESS_VERSION = "winsorize6mv_hp0.05_v2"  # đổi khi sửa nội dung preprocess()

# ===================== QC TÍN HIỆU =================
# Rà soát chuyển đạo flatline/rớt điện cực (mục 6b) chỉ BÁO CÁO, không tự loại. Nếu xác nhận
# là lỗi ghi thật, thêm record_stem vào danh sách dưới rồi chạy lại từ đầu notebook -- dùng lại
# đúng cơ chế loại bỏ đã kiểm chứng của KNOWN_BROKEN_RECORDS, không thêm luồng xử lý song song.
QC_EXTRA_EXCLUDE = []                    # điền record_stem sau khi review báo cáo QC mục 6b
FLATLINE_STD_THRESHOLD_MV = 0.02         # std một chuyển đạo dưới ngưỡng này (sau lọc) -> nghi ngờ

# ===================== THAM SỐ AUGMENTATION (chỉ áp cho tập train) ======
AUG_ENABLE = True
AUG_MAX_SHIFT = 20          # dịch thời gian tối đa (mẫu, ~40ms @ 500Hz) — mô phỏng jitter giữa các lần ghi
AUG_NOISE_STD = 0.02        # nhiễu Gaussian biên độ thấp, đo trên tín hiệu ĐÃ chuẩn hoá (std≈1)
AUG_GAIN_JITTER = 0.05      # hệ số gain ngẫu nhiên ±5%/chuyển đạo — mô phỏng lệch tiếp xúc điện cực
AUG_OP_PROB = 0.5           # xác suất áp mỗi phép augment riêng biệt cho mỗi mẫu (3 augment gốc)
AUG_POWERLINE_PROB = 0.3    # xác suất chồng nhiễu điện lưới
AUG_POWERLINE_HZ = 50.0     # tần số lưới điện Việt Nam
AUG_POWERLINE_STD = 0.03    # biên độ nhiễu điện lưới, đo trên tín hiệu ĐÃ chuẩn hoá (std≈1)
AUG_LEAD_DROPOUT_PROB = 0.03  # xác suất một chuyển đạo bị thay bằng nhiễu thấp (mô phỏng rớt điện cực)

# ===================== THAM SỐ LOSS (Focal + label smoothing) ===========
FOCAL_GAMMA = 2.0            # 0 = tắt hoàn toàn phần điều chỉnh focal, quay về BCE có trọng số lớp
LABEL_SMOOTH_EPS = 0.02      # 0 = tắt label smoothing

# ===================== THAM SỐ HUẤN LUYỆN =========================
VAL_SIZE = 0.15                                 # chia theo bệnh nhân 85/15 trong POOL để early-stop
TEST_SIZE = 0.15                                # 15% bệnh nhân giữ riêng làm TEST, tách trước K-Fold
LR, WEIGHT_DECAY = 1e-3, 1e-4
EARLY_STOP_PATIENCE, LR_PATIENCE = 10, 5
TARGET_LABEL = "STEMI"
CLASS_NAME = "STEMI"
K_FOLDS = 5

# --- [v3] LR riêng cho kiến trúc dùng LayerNorm/attention -------------------
# Chẩn đoán từ v2: ConvNeXtV2_1D có best_epoch dao động 7->28 giữa các fold (std 7,55 -- gấp
# ~3 lần mọi model khác) và AUPRC OOF thấp nhất đoàn (0,5911 so với 0,64-0,67). Đây là dấu
# hiệu kinh điển của LR quá cao với mạng LayerNorm+GELU+GRN: AdamW ở 1e-3 phá thống kê chuẩn
# hoá ngay trong vài epoch đầu. Hạ về 3e-4 -- đúng bậc LR ConvNeXt gốc dùng cho batch nhỏ.
MODEL_LR_OVERRIDE = {"ConvNeXtV2_1D": 3e-4, "CNNTransformer": 3e-4}

# --- [v3] Warmup tuyến tính --------------------------------------------------
# Trong WARMUP_EPOCHS epoch đầu, LR tăng tuyến tính tới base_lr; ReduceLROnPlateau chỉ bắt
# đầu hoạt động SAU giai đoạn này. Ổn định gradient những epoch đầu cho mọi model, đặc biệt
# nhóm LayerNorm/attention.
WARMUP_EPOCHS = 3

# --- [v3] Danh sách kiến trúc ------------------------------------------------
# 8 kiến trúc của v2 + 2 kiến trúc MỚI có inductive bias khác hẳn. Lý do: trong 8 model cũ có
# tới 4 cái (ResNet1D, SEResNet1D, XResNet1D, AiTiAMI) đều là residual-CNN kernel 5-7, nên lỗi
# của chúng tương quan cao và ensemble gần như không lợi thêm. TCN1D (conv giãn nở, trường thu
# nhận tăng theo cấp số nhân) và CNNTransformer (tự chú ý toàn cục thay vì tuần tự như BiLSTM)
# tạo lỗi khác kiểu -- đó mới là thứ ensemble khai thác được.
# Muốn giảm thời gian chạy: xoá bớt tên khỏi danh sách này (checkpoint các model còn lại vẫn
# dùng lại được vì fingerprint tính riêng cho từng model).
CANDIDATES = ["PlainCNN", "XResNet1D", "ConvNeXtV2_1D", "AiTiAMI",
              "ResNet1D", "SEResNet1D", "CNN+BiLSTM", "InceptionTime1D",
              "TCN1D", "CNNTransformer"]

# ===================== NGƯỠNG VẬN HÀNH LÂM SÀNG =========================
TARGET_SENSITIVITY = 0.91    # chính sách triage: rule-out tại Sensitivity >= 91%, chốt trên POOL (OOF)
SENS_TARGETS = [0.91, 0.92, 0.93, 0.94, 0.95]   # bảng quét điểm vận hành ở mục 21
ECE_BINS = 10

# --- [v3] Biên an toàn khi chuyển ngưỡng POOL -> TEST ------------------------
# Ngưỡng chốt trên POOL không bao giờ cho đúng Sensitivity đó trên TEST: TEST chỉ có ~217 ca
# dương nên Sensitivity dao động ~±2 điểm % thuần do cỡ mẫu. v2 đã dính đúng bẫy này (chốt
# 91% trên POOL -> chỉ đạt 88,9% trên TEST = VI PHẠM chính sách đã đặt ra). Cách sửa: đo độ
# lệch chuẩn của Sensitivity giữa 5 fold tại cùng một ngưỡng (ước lượng THUẦN POOL cho mức dao
# động trên một tập cỡ tương tự), rồi nhắm cao hơn mục tiêu z lần độ lệch đó.
# z = 1.28 -> xác suất ~90% đạt được mục tiêu thật trên TEST.
SENS_MARGIN_Z = 1.28
USE_SENS_MARGIN = True       # False = quay về đúng hành vi v2 (không biên), để đối chiếu
# Trần cho mục tiêu sau khi cộng biên. Bắt buộc phải có: nếu Sensitivity dao động rất mạnh
# giữa các fold, target + z*sd có thể tiến sát 1,0 và ngưỡng tụt xuống mức dự đoán MỌI ca là
# dương (Specificity = 0) -- hỏng âm thầm mà bảng vẫn in ra bình thường. Chạm trần sẽ được
# cảnh báo tường minh ở mục 19.
SENS_MARGIN_MAX_TARGET = 0.97

# ===================== THAM SỐ ENSEMBLE & THỐNG KÊ (v3) =================
# Predictor chính khi báo cáo TEST:
#   "bagging" = trung bình 5 checkpoint fold/kiến trúc (khuyến nghị)
#   "final"   = 1 checkpoint train trên 100% POOL (cách v2 dùng)
# Bagging vừa mạnh hơn (đo trên chính bộ dữ liệu này: +0,03..+0,05 AUPRC cho model đơn) vừa
# khớp thang xác suất với OOF -- nên ngưỡng chốt từ POOL chuyển sang TEST đúng hơn hẳn.
PRIMARY_PREDICTOR = "bagging"
GREEDY_ITERS = 40            # số vòng lặp Caruana greedy ensemble selection
STACK_C = 1.0                # nghịch đảo cường độ regularization của stacking logistic
N_BOOTSTRAP = 2000           # số lần bootstrap (lấy mẫu THEO BỆNH NHÂN, giống Phần I báo cáo)

# --- Mốc so sánh: số liệu STEMI đang có trong Bao_cao_nghien_cuu.docx (mục 2.2.1) ---
# CHỈ dùng để in bảng chênh lệch cuối notebook. KHÔNG có bước nào đọc con số này để ra quyết
# định -- mọi lựa chọn model/ngưỡng/ensemble đều chốt trên POOL trước khi chạm vào TEST.
REPORT_BASELINE = {
    "AUROC": 0.9517, "AUPRC": 0.7435, "Sensitivity": 0.9447, "Specificity": 0.8231,
    "PPV": 0.3198, "NPV": 0.9941, "F1": 0.4779, "TP": 205, "FN": 12, "FP": 436, "TN": 2028,
}
REPORT_BASELINE_NOTE = ("Average Ensemble 8 model, checkpoint FINAL, ngưỡng 0.2417 "
                        "-- TEST 2.681 bản ghi / 217 dương")

# Số ca trong paper gốc (toàn bộ 19.955 bản ghi); ta chỉ có nhãn cho ~90% nên dung sai ±10%
PAPER_COUNTS = {"STEMI": 1513, "NSTEMI": 1274, "UA": 6197}
SANITY_TOL = 0.10

# Hai bản ghi lỗi đọc thật ở nguồn (wfdb báo ValueError ngay cả khi đọc đúng số mẫu thực có
# trong .dat) — không phải trường hợp thiếu vài mẫu vô hại có thể suy luận lại. Loại ngay ở
# bước tạo nhãn, trước khi chia tập/tính cache, để không rò rỉ ảnh hưởng vào bất kỳ tập con nào.
KNOWN_BROKEN_RECORDS = ["03228", "14262"]

# ===================== SUY RA ==========================
GPU_AVAILABLE = torch.cuda.is_available()
DEVICE = torch.device("cuda" if GPU_AVAILABLE else "cpu")
DEBUG_LIMIT = 200
# [v3] 30 -> 40: ở v2, trung vị best_epoch là 23-27 và HAI model chạm trần 30 (ResNet1D,
# CNN+BiLSTM) -- tức chúng vẫn còn đang cải thiện khi bị cắt. Early stop (patience 10) vẫn
# dừng sớm phần lớn model nên chi phí thực tế tăng ít hơn tỷ lệ 40/30.
EPOCHS = 3 if RUN_MODE == "debug" else 40
BATCH_SIZE = 8 if RUN_MODE == "debug" else (64 if GPU_AVAILABLE else 16)
USE_AMP = GPU_AVAILABLE
N_FOLDS_RUN = 1 if RUN_MODE == "debug" else K_FOLDS

assert PRIMARY_PREDICTOR in ("bagging", "final")

print("Môi trường :", "Google Colab" if IS_COLAB else "máy cá nhân")
print("Thiết bị   :", DEVICE)
print("Chế độ     :", RUN_MODE, f"({EPOCHS} epoch, batch {BATCH_SIZE}, warmup {WARMUP_EPOCHS})")
print("Nhãn đích  :", TARGET_LABEL)
print(f"Ứng viên   : {len(CANDIDATES)} kiến trúc -- " + ", ".join(CANDIDATES))
print("K-Fold     :", f"{K_FOLDS} fold, chạy {N_FOLDS_RUN} fold ở chế độ {RUN_MODE}")
print("Checkpoint :", MODEL_DIR)
print("Predictor  :", PRIMARY_PREDICTOR, "| LR override:", MODEL_LR_OVERRIDE or "(không)")
print("Ngưỡng đích:", f"Sensitivity >= {TARGET_SENSITIVITY:.0%}",
      f"(chốt trên POOL, biên an toàn z={SENS_MARGIN_Z})" if USE_SENS_MARGIN else "(không biên)")
print(f"Bandpass   : {BP_LOW}-{BP_HIGH} Hz, bậc {BP_ORDER}, zero-phase (filtfilt)")
print(f"QC flatline: ngưỡng std < {FLATLINE_STD_THRESHOLD_MV} mV | loại thêm: {QC_EXTRA_EXCLUDE or '(rỗng)'}")

## 3. Chuẩn bị dữ liệu

In [ ]:
import time

for d in [CACHE_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

assert DATA_ROOT.exists(), (
    f"Không thấy {DATA_ROOT} -- kiểm tra lại thư mục 'datasets/' có nằm cùng cấp với notebook "
    "này không (repo ECG-experiment)."
)
print("DATA_ROOT:", DATA_ROOT, "| tồn tại:", DATA_ROOT.exists())

## 4. Seed & GPU

In [ ]:
import random

import numpy as np

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Input cố định (12 x 5000) nên cuDNN chọn thuật toán nhanh nhất một lần rồi giữ nguyên.
# Đánh đổi: epoch đầu chậm hơn vài giây, kết quả không tái lập chính xác 100% giữa các máy.
torch.backends.cudnn.benchmark = GPU_AVAILABLE

print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if GPU_AVAILABLE else "không có")
if not GPU_AVAILABLE:
    print("\n" + "!" * 70)
    print("!! KHÔNG PHÁT HIỆN GPU CUDA -- notebook này thiết kế cho máy có GPU (RTX 2000 Ada).")
    print("!! Kiểm tra lại driver/CUDA/bản torch trước khi Run All, nếu không epoch 'full' sẽ")
    print("!! mất hàng chục phút thay vì vài chục giây.")
    print("!" * 70)

## 5. Dữ liệu và nhãn

Nạp bảng nhãn CSV, đối chiếu số ca dương với con số công bố trong paper gốc (sanity check),
rồi loại hai bản ghi lỗi đọc thật ở nguồn.

In [ ]:
import pandas as pd

pd.set_option("display.width", 200)


def find_dir(root: Path, names):
    wanted = {n.lower() for n in names}
    for c in sorted(root.iterdir()):
        if c.is_dir() and c.name.lower() in wanted:
            return c
    for c in root.rglob("*"):
        if c.is_dir() and c.name.lower() in wanted:
            return c
    return None


RAW_DIR = find_dir(DATA_ROOT, ["row_data", "raw_data"])
CSV_DIR = find_dir(DATA_ROOT, ["CSV", "csv"])
assert RAW_DIR and CSV_DIR, f"Không thấy row_data/ hoặc CSV/ trong {DATA_ROOT}"

df_raw = pd.read_csv(CSV_DIR / "train.csv")
df_raw["record_stem"] = df_raw["ecg_row_record"].astype(str).str.replace(".dat", "", regex=False)

COL_PATIENT, COL_AGE, COL_GENDER = "Patient_id", "age", "gender"
print(f"{len(df_raw):,} bản ghi | {df_raw[COL_PATIENT].nunique():,} bệnh nhân | "
      f"{len(list(RAW_DIR.glob('*.dat'))):,} file .dat")

# --- Sanity check so với paper ---
print(f"\n{'nhãn':<8}{'đếm được':>11}{'paper':>9}{'lệch':>9}")
ok_all = True
for name, expected in PAPER_COUNTS.items():
    got = int(df_raw[name].sum())
    delta = (got - expected) / expected
    ok_all &= abs(delta) <= SANITY_TOL
    print(f"{name:<8}{got:>11,}{expected:>9,}{delta:>8.1%}   {'OK' if abs(delta) <= SANITY_TOL else 'LỆCH'}")
if not ok_all:
    print("\n!! CẢNH BÁO: số ca lệch quá ±10% so với paper — kiểm tra lại nguồn dữ liệu.")

# --- Nhãn đích ---
df_all = df_raw.copy()
df_all[TARGET_LABEL] = df_all["STEMI"].astype(int)
print("Nhãn =", TARGET_LABEL, "(lấy trực tiếp từ cột cùng tên)")

# --- Loại bản ghi lỗi đọc thật ở nguồn + bản ghi QC thủ công (mục 6b, nếu có) ---
_exclude_list = list(dict.fromkeys(KNOWN_BROKEN_RECORDS + QC_EXTRA_EXCLUDE))  # gộp, giữ thứ tự, bỏ trùng
_broken_mask = df_all["record_stem"].isin(_exclude_list)
if _broken_mask.any():
    print(f"\nLoại {int(_broken_mask.sum())} bản ghi (lỗi đọc thật + QC thủ công mục 6b): "
          f"{df_all.loc[_broken_mask, 'record_stem'].tolist()}")
    display(df_all.loc[_broken_mask, ["record_stem", COL_PATIENT, TARGET_LABEL, "NSTEMI", "UA"]])
    df_all = df_all[~_broken_mask].reset_index(drop=True)
else:
    print("\nKhông thấy bản ghi lỗi đã biết trong dữ liệu hiện tại (nguồn có thể đã đổi) "
          "-- kiểm tra lại KNOWN_BROKEN_RECORDS/QC_EXTRA_EXCLUDE nếu nghi ngờ.")

n_pos = int(df_all[TARGET_LABEL].sum())
print(f"\nNhãn {TARGET_LABEL}: {n_pos:,} dương ({n_pos / len(df_all):.2%}) / "
      f"{len(df_all) - n_pos:,} âm")

if RUN_MODE == "debug" and len(df_all) > DEBUG_LIMIT:
    from sklearn.model_selection import train_test_split
    df, _ = train_test_split(df_all, train_size=DEBUG_LIMIT,
                             stratify=df_all[TARGET_LABEL], random_state=SEED)
    df = df.reset_index(drop=True)
else:
    df = df_all.reset_index(drop=True)

n_pos = int(df[TARGET_LABEL].sum())
print(f"Dùng {len(df):,} bản ghi | {n_pos:,} dương ({n_pos / len(df):.2%})")
assert 0 < n_pos < len(df), "Tập chỉ có một lớp — không train được."

## 6. Tiền xử lý tín hiệu & cache

Bandpass Butterworth (0,5–40 Hz) rồi **winsorize** biên độ ở ±6mV — ngưỡng này chặn artifact/
nhiễu điện cực (biên độ vượt xa QRS sinh lý bình thường ~1–3mV) mà không cắt cụt hình dạng
sóng bình thường, vì áp dụng bằng `np.clip` (giữ nguyên hình dạng phần trong ngưỡng, chỉ kẹp
phần đuôi cực đoan) chứ không co giãn lại toàn bộ tín hiệu. Winsorize thực hiện **sau** bandpass
và **trước** khi tính thống kê z-score, để vài giá trị cực đoan của từng bản ghi không kéo lệch
độ lệch chuẩn dùng chuẩn hoá bản ghi đó.

Cache tín hiệu đã tiền xử lý ra `.npy` (float16) để không phải lọc lại mỗi lần chạy; tên file
cache khoá theo hash của `PREPROCESS_VERSION` + danh sách bản ghi, nên sửa nội dung
`preprocess()` (đổi `PREPROCESS_VERSION`) sẽ tự động build cache mới thay vì âm thầm dùng cache
cũ đã lỗi thời.

In [ ]:
import hashlib
import json

from scipy.signal import butter, filtfilt

_B, _A = butter(BP_ORDER, [BP_LOW / (FS / 2), BP_HIGH / (FS / 2)], btype="band")
TRUNCATED = []


def load_signal(stem: str) -> np.ndarray:
    path = str(RAW_DIR / stem)
    try:
        rec = wfdb.rdrecord(path)
    except ValueError:                     # .dat ngắn hơn header khai báo -> đọc đúng số mẫu thực có
        n_sig = int((RAW_DIR / f"{stem}.hea").read_text().splitlines()[0].split()[1])
        n = (RAW_DIR / f"{stem}.dat").stat().st_size // (n_sig * 2)
        rec = wfdb.rdrecord(path, sampto=n)
        TRUNCATED.append(stem)
    return np.asarray(rec.p_signal, dtype=np.float32).T


def preprocess(sig: np.ndarray) -> np.ndarray:
    sig = np.nan_to_num(sig, nan=0.0, posinf=0.0, neginf=0.0)
    if sig.shape[1] < SIGNAL_LEN:
        sig = np.pad(sig, ((0, 0), (0, SIGNAL_LEN - sig.shape[1])))
    filtered = filtfilt(_B, _A, sig[:, :SIGNAL_LEN], axis=1)
    winsorized = np.clip(filtered, -WINSORIZE_MV, WINSORIZE_MV)
    return np.ascontiguousarray(winsorized, dtype=np.float32)


records = df["record_stem"].tolist()
_hash = hashlib.md5((PREPROCESS_VERSION + "|" + "|".join(records)).encode()).hexdigest()[:8]
CACHE_NPY = CACHE_DIR / f"sig_{RUN_MODE}_n{len(df)}_{_hash}.npy"
CACHE_META = CACHE_DIR / f"sig_{RUN_MODE}_n{len(df)}_{_hash}.meta.json"


def _complete(meta_p: Path, npy_p: Path) -> bool:
    if not (meta_p.exists() and npy_p.exists()):
        return False
    try:
        return json.loads(meta_p.read_text()).get("n_done", 0) >= len(records)
    except (OSError, ValueError):
        return False


def build_cache() -> np.ndarray:
    if _complete(CACHE_META, CACHE_NPY):
        print("Cache đã đầy đủ:", CACHE_NPY.name)
        return np.load(CACHE_NPY, mmap_mode="r")

    meta = json.loads(CACHE_META.read_text()) if CACHE_META.exists() else {}
    start = int(meta.get("n_done", 0)) if CACHE_NPY.exists() else 0
    if start:
        arr = np.lib.format.open_memmap(CACHE_NPY, mode="r+")
        print(f"Build tiếp từ {start}/{len(records)}")
    else:
        print(f"Build cache {len(records):,} bản ghi "
              f"(~{len(records) * NUM_LEADS * SIGNAL_LEN * 2 / 1024 ** 3:.2f} GB)")
        arr = np.lib.format.open_memmap(CACHE_NPY, mode="w+", dtype=np.float16,
                                        shape=(len(records), NUM_LEADS, SIGNAL_LEN))

    t0 = time.time()
    step = max(1, len(records) // 8)
    for i in range(start, len(records)):
        arr[i] = preprocess(load_signal(records[i])).astype(np.float16)
        if (i + 1) % step == 0 or i + 1 == len(records):
            arr.flush()
            CACHE_META.write_text(json.dumps({"n_done": i + 1}))
            el = max(time.time() - t0, 1e-6)
            done = i + 1 - start
            print(f"  {i + 1:>6,}/{len(records):,}  {done / el:5.0f} rec/s  "
                  f"ETA {(len(records) - i - 1) / (done / el):4.0f}s")
    del arr
    print(f"Xong trong {time.time() - t0:.0f}s")
    return np.load(CACHE_NPY, mmap_mode="r")


CACHE = build_cache()
print("Cache:", CACHE.shape, CACHE.dtype)
if TRUNCATED:
    print(f"Bản ghi bị cắt ngắn (đã đọc đúng số mẫu thực có, phần thiếu được zero-pad): {TRUNCATED}")

## 6b. QC: rà soát chuyển đạo flatline / rớt điện cực (mới)

Quét std của từng chuyển đạo trên toàn bộ cache đã tiền xử lý. Đây chỉ là **báo cáo**, không
tự động loại bỏ bản ghi nào — winsorize (mục 6) chỉ xử lý biên độ outlier lớn, không bắt được
trường hợp một chuyển đạo phẳng gần 0 do điện cực rớt (biên độ vẫn nằm trong khoảng cho phép).
Nếu sau khi review thấy đúng là lỗi ghi, thêm `record_stem` vào `QC_EXTRA_EXCLUDE` (mục 2) rồi
chạy lại từ đầu — dùng lại đúng cơ chế loại bỏ đã kiểm chứng của `KNOWN_BROKEN_RECORDS`.

In [ ]:
def scan_flatline(cache, chunk=256, threshold_mv=FLATLINE_STD_THRESHOLD_MV):
    """Std của từng chuyển đạo cho từng bản ghi, tính theo batch để không load hết cache vào RAM."""
    n = cache.shape[0]
    min_lead_std = np.empty(n, dtype=np.float32)
    flat_lead_idx = np.empty(n, dtype=np.int16)
    for i in range(0, n, chunk):
        b = np.asarray(cache[i:i + chunk], dtype=np.float32)   # (chunk, lead, time)
        lead_std = b.std(axis=2)                                # (chunk, lead)
        min_lead_std[i:i + chunk] = lead_std.min(axis=1)
        flat_lead_idx[i:i + chunk] = lead_std.argmin(axis=1)
    return min_lead_std, flat_lead_idx


LEAD_NAMES = ["I", "II", "III", "aVR", "aVL", "aVF",
              "V1", "V2", "V3", "V4", "V5", "V6"][:NUM_LEADS]

_min_std, _flat_lead = scan_flatline(CACHE)
_flag = _min_std < FLATLINE_STD_THRESHOLD_MV
print(f"QC flatline: ngưỡng std < {FLATLINE_STD_THRESHOLD_MV} mV -> "
      f"{int(_flag.sum())}/{len(df):,} bản ghi nghi ngờ")

if _flag.any():
    QC_FLATLINE_REPORT = pd.DataFrame({
        "record_stem": df.loc[_flag, "record_stem"].values,
        "patient": df.loc[_flag, COL_PATIENT].values,
        "lead_nghi_ngo": [LEAD_NAMES[i] for i in _flat_lead[_flag]],
        "std_thap_nhat_mV": _min_std[_flag],
        TARGET_LABEL: df.loc[_flag, TARGET_LABEL].values,
    }).sort_values("std_thap_nhat_mV").reset_index(drop=True)
    display(QC_FLATLINE_REPORT)
    print("\n-- Đây chỉ là RÀ SOÁT, chưa tự động loại. Nếu xác nhận là lỗi ghi thật,")
    print("   thêm record_stem vào QC_EXTRA_EXCLUDE ở mục 2 rồi chạy lại từ đầu notebook.")
else:
    QC_FLATLINE_REPORT = pd.DataFrame(
        columns=["record_stem", "patient", "lead_nghi_ngo", "std_thap_nhat_mV", TARGET_LABEL])
    print("Không phát hiện chuyển đạo flatline nào dưới ngưỡng.")

print(f"\nPhân phối std nhỏ nhất/bản ghi: min={_min_std.min():.4f}  "
      f"p1={np.percentile(_min_std, 1):.4f}  median={np.median(_min_std):.4f} mV")


## 7. Chia POOL / TEST theo bệnh nhân

Trích 15% **bệnh nhân** làm TEST giữ riêng trước khi làm bất cứ điều gì khác (K-Fold, chuẩn
hoá, chọn ngưỡng...). TEST không được chạm tới cho đến bảng kết quả cuối cùng.

In [ ]:
from sklearn.model_selection import train_test_split

y = df[TARGET_LABEL].values.astype(np.float32)

pat_all = df.groupby(COL_PATIENT)[TARGET_LABEL].max().reset_index()
pat_pool, pat_test = train_test_split(pat_all, test_size=TEST_SIZE,
                                      stratify=pat_all[TARGET_LABEL], random_state=SEED + 200)
TEST_PATIENTS = set(pat_test[COL_PATIENT])
IS_TEST = df[COL_PATIENT].isin(TEST_PATIENTS).to_numpy()
POOL_IDX = df.index[~IS_TEST].to_numpy()
TEST_IDX = df.index[IS_TEST].to_numpy()

assert not (set(df.loc[POOL_IDX, COL_PATIENT]) & TEST_PATIENTS), "RÒ RỈ: bệnh nhân TEST lọt vào POOL"
assert len(POOL_IDX) + len(TEST_IDX) == len(df), "POOL + TEST không phủ đúng toàn bộ df"

display(pd.DataFrame({
    "tập": ["POOL (train + K-Fold)", "TEST (giữ riêng)"],
    "bệnh nhân": [len(pat_pool), len(pat_test)],
    "bản ghi": [len(POOL_IDX), len(TEST_IDX)],
    "dương": [int(y[POOL_IDX].sum()), int(y[TEST_IDX].sum())],
    "tỷ lệ dương": [f"{y[POOL_IDX].mean():.2%}", f"{y[TEST_IDX].mean():.2%}"],
}))


def norm_stats(idx, chunk=256):
    """z-score fit trên đúng tập con truyền vào (dùng riêng cho từng fold train)."""
    order = np.sort(np.asarray(idx))
    s = np.zeros(NUM_LEADS)
    ss = np.zeros(NUM_LEADS)
    cnt = 0
    for i in range(0, len(order), chunk):
        b = np.asarray(CACHE[order[i:i + chunk]], dtype=np.float64)
        s += b.sum(axis=(0, 2))
        ss += (b ** 2).sum(axis=(0, 2))
        cnt += b.shape[0] * b.shape[2]
    m = s / cnt
    return m.astype(np.float32), np.sqrt(np.maximum(ss / cnt - m ** 2, 1e-12)).astype(np.float32)


mean_pool, std_pool = norm_stats(POOL_IDX)
print(f"\nz-score fit trên {len(POOL_IDX):,} bản ghi POOL (dùng cho model FINAL, mục 13)")

## 8. Chia K-Fold theo bệnh nhân trong POOL

In [ ]:
from sklearn.model_selection import StratifiedKFold

pat_tbl = df.loc[POOL_IDX].groupby(COL_PATIENT)[TARGET_LABEL].max().reset_index()
POS_RATE_ALL = float(y[POOL_IDX].mean())

FOLDS = {}          # k -> (train_idx, val_idx), chỉ trong POOL_IDX
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
for k, (_, va_pos) in enumerate(skf.split(pat_tbl, pat_tbl[TARGET_LABEL])):
    va_pats = set(pat_tbl.iloc[va_pos][COL_PATIENT])
    in_va_pool = df.loc[POOL_IDX, COL_PATIENT].isin(va_pats).to_numpy()
    FOLDS[k] = (POOL_IDX[~in_va_pool], POOL_IDX[in_va_pool])

# --- Kiểm tra không rò rỉ + phủ đúng POOL một lần ---
seen = set()
for k in range(K_FOLDS):
    pats_k = set(df.loc[FOLDS[k][1], COL_PATIENT])
    assert not (seen & pats_k), f"RÒ RỈ: bệnh nhân nằm ở hai fold (fold {k})"
    assert not (pats_k & TEST_PATIENTS), f"RÒ RỈ: bệnh nhân TEST lọt vào fold {k}"
    seen |= pats_k
    assert not (set(df.loc[FOLDS[k][0], COL_PATIENT]) & pats_k), \
        f"RÒ RỈ: bệnh nhân vừa ở train vừa ở val (fold {k})"
cover = np.concatenate([FOLDS[k][1] for k in range(K_FOLDS)])
assert len(cover) == len(POOL_IDX) and len(set(cover)) == len(POOL_IDX), \
    "Các fold không phủ đúng POOL_IDX một lần"
print(f"Giao nhau bệnh nhân giữa các fold: RỖNG -> OK | TEST không lọt vào fold nào -> OK")

rows, worst = [], 0.0
for k, (tr, va) in FOLDS.items():
    rate = float(y[va].mean())
    worst = max(worst, abs(rate - POS_RATE_ALL))
    rows.append({"fold": k, "bệnh nhân val": df.loc[va, COL_PATIENT].nunique(),
                 "bản ghi train": len(tr), "bản ghi val": len(va),
                 "dương val": int(y[va].sum()), "tỷ lệ dương val": rate,
                 "lệch (điểm %)": (rate - POS_RATE_ALL) * 100})
fold_df = pd.DataFrame(rows)
display(fold_df.style.format({"tỷ lệ dương val": "{:.2%}", "lệch (điểm %)": "{:+.2f}"}))
assert worst <= 0.02, f"Fold lệch tỷ lệ dương tới {worst:.2%} > 2 điểm %"
print(f"Tỷ lệ dương POOL {POS_RATE_ALL:.2%} | lệch lớn nhất giữa các fold {worst * 100:.2f} điểm % -> OK")

## 9. Dataset & DataLoader

Augmentation (`augment=True`) chỉ áp cho **tập train của từng fold**, không bao giờ áp cho
tập validation/POOL-OOF/TEST — nếu không, chỉ số đánh giá sẽ đo trên tín hiệu đã bị nhiễu
nhân tạo thay vì tín hiệu thật. Ba phép biến đổi, mỗi phép có xác suất áp riêng
(`AUG_OP_PROB`), đều giữ nguyên hình dạng đoạn ST/QRS (không distort đặc trưng chẩn đoán):

- **Dịch thời gian** (≤ `AUG_MAX_SHIFT` mẫu, đệm bằng giá trị biên chứ không cuộn vòng để
  tránh tạo bước nhảy giả ở điểm nối) — mô phỏng lệch thời điểm bắt đầu ghi giữa các lần đo.
- **Nhiễu Gaussian biên độ thấp** trên tín hiệu đã chuẩn hoá — mô phỏng nhiễu điện tử.
- **Gain jitter riêng từng chuyển đạo** (±`AUG_GAIN_JITTER`) — mô phỏng tiếp xúc điện cực
  không hoàn toàn đồng nhất giữa các lần ghi.

In [ ]:
from torch.utils.data import DataLoader, Dataset


def _time_shift(x: np.ndarray, max_shift: int) -> np.ndarray:
    shift = np.random.randint(-max_shift, max_shift + 1)
    if shift == 0:
        return x
    if shift > 0:
        return np.pad(x, ((0, 0), (shift, 0)), mode="edge")[:, :x.shape[1]]
    return np.pad(x, ((0, 0), (0, -shift)), mode="edge")[:, -x.shape[1]:]


def _powerline_noise(x: np.ndarray, fs: int, hz: float, std: float) -> np.ndarray:
    """Mô phỏng nhiễu điện lưới (50Hz tại VN) chồng lên MỌI chuyển đạo cùng lúc (giống nhiễu
    thật, vốn lan theo dây/đất chung) -- biên độ và pha ngẫu nhiên mỗi mẫu để tránh mô hình
    học vẹt một pha cố định thay vì học cách bỏ qua nhiễu."""
    t = np.arange(x.shape[1]) / fs
    phase = np.random.uniform(0, 2 * np.pi)
    amp = np.random.uniform(0.3, 1.0) * std
    noise = (amp * np.sin(2 * np.pi * hz * t + phase)).astype(np.float32)
    return x + noise[None, :]


def _lead_dropout(x: np.ndarray) -> np.ndarray:
    """Mô phỏng MỘT chuyển đạo bị rớt/tiếp xúc kém: thay bằng nhiễu biên độ thấp gần 0,
    không đụng tới các chuyển đạo còn lại. Xác suất áp thấp (AUG_LEAD_DROPOUT_PROB) vì đây
    là hỏng nặng, không nên xảy ra thường xuyên trong lúc train."""
    x = x.copy()
    lead = np.random.randint(0, x.shape[0])
    x[lead] = np.random.normal(0.0, 0.05, size=x.shape[1]).astype(np.float32)
    return x


class ECGDataset(Dataset):
    """z-score phải fit lại trên tập train CỦA TỪNG FOLD, không dùng chung một bộ thống kê
    toàn cục -- nếu không thì thống kê của fold này rò rỉ sang fold kia. `augment=True` chỉ
    được bật cho DataLoader huấn luyện của fold đó.
    """

    def __init__(self, indices, labels, mean, std, augment=False):
        self.indices = np.asarray(indices)
        self.labels = np.asarray(labels, dtype=np.float32)
        self.mean = np.asarray(mean).reshape(-1, 1)
        self.std = np.asarray(std).reshape(-1, 1)
        self.augment = augment and AUG_ENABLE

    def __len__(self):
        return len(self.indices)

    def _apply_augment(self, x: np.ndarray) -> np.ndarray:
        # --- 3 augmentation gốc (giữ nguyên từ v1) ---
        if np.random.rand() < AUG_OP_PROB:
            x = _time_shift(x, AUG_MAX_SHIFT)
        if np.random.rand() < AUG_OP_PROB:
            x = x + np.random.normal(0.0, AUG_NOISE_STD, size=x.shape).astype(np.float32)
        if np.random.rand() < AUG_OP_PROB:
            gain = np.random.uniform(1 - AUG_GAIN_JITTER, 1 + AUG_GAIN_JITTER,
                                     size=(NUM_LEADS, 1)).astype(np.float32)
            x = x * gain
        # --- 2 augmentation mới (v2): mô phỏng nhiễu thực tế lâm sàng ---
        if np.random.rand() < AUG_POWERLINE_PROB:
            x = _powerline_noise(x, FS, AUG_POWERLINE_HZ, AUG_POWERLINE_STD)
        if np.random.rand() < AUG_LEAD_DROPOUT_PROB:
            x = _lead_dropout(x)
        return x

    def __getitem__(self, i):
        x = np.asarray(CACHE[self.indices[i]], dtype=np.float32)
        x = (x - self.mean) / self.std
        if self.augment:
            x = self._apply_augment(x)
        return torch.from_numpy(np.ascontiguousarray(x)), torch.tensor(self.labels[i])


# Windows tạo worker bằng spawn (mỗi worker nạp lại cả notebook) nên để 0; Linux dùng fork -> 2
NUM_WORKERS = 0 if os.name == "nt" else 2
_extra = dict(persistent_workers=True, prefetch_factor=4) if NUM_WORKERS else {}


def make_loaders(tr_idx, va_idx, mean, std):
    ds_tr = ECGDataset(tr_idx, y[tr_idx], mean, std, augment=True)
    ds_va = ECGDataset(va_idx, y[va_idx], mean, std, augment=False)
    tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=NUM_WORKERS, pin_memory=GPU_AVAILABLE, **_extra)
    va = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=GPU_AVAILABLE, **_extra)
    return tr, va


_tr_demo, _va_demo = make_loaders(*FOLDS[0], *norm_stats(FOLDS[0][0]))
xb, yb = next(iter(_va_demo))
print(f"batch: {tuple(xb.shape)} {xb.dtype} | nhãn {tuple(yb.shape)} | "
      f"{len(_tr_demo)} batch train, {len(_va_demo)} batch val (fold 0, minh hoạ)")
del _tr_demo, _va_demo

## 10. Hàm tính metric

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (average_precision_score, brier_score_loss, confusion_matrix,
                             precision_recall_fscore_support, roc_auc_score, roc_curve)

plt.rcParams["figure.dpi"] = 110


def _div(a, b):
    return float(a) / float(b) if b else float("nan")


def compute_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int).ravel()
    y_prob = np.asarray(y_prob, dtype=np.float64).ravel()
    y_pred = (y_prob >= threshold).astype(int)
    two = len(np.unique(y_true)) == 2
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "auroc": roc_auc_score(y_true, y_prob) if two else float("nan"),
        "auprc": average_precision_score(y_true, y_prob) if two else float("nan"),
        "sensitivity": _div(tp, tp + fn), "specificity": _div(tn, tn + fp),
        "ppv": _div(tp, tp + fp), "npv": _div(tn, tn + fn),
        "f1": _div(2 * tp, 2 * tp + fp + fn),
        "accuracy": _div(tp + tn, tp + tn + fp + fn),
        "brier": float(brier_score_loss(y_true, y_prob)) if two else float("nan"),
        "threshold": float(threshold),
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
        "n": int(len(y_true)), "n_pos": int(y_true.sum()),
    }


def threshold_at_sensitivity(y_true, y_prob, target_sens):
    """Ngưỡng LỚN NHẤT sao cho Sensitivity thực tế vẫn >= target_sens -- dò trực tiếp trên điểm
    số của các ca dương (không nội suy tuyến tính trên ROC dạng bậc thang, vốn có thể cho kết
    quả tụt dưới mục tiêu)."""
    y_true = np.asarray(y_true).astype(int)
    pos_scores = np.sort(np.asarray(y_prob)[y_true == 1])[::-1]
    n_pos = len(pos_scores)
    k = min(int(np.ceil(target_sens * n_pos)), n_pos)
    return float(pos_scores[k - 1]) if k > 0 else 1.0


def style_df(df_, pct_cols):
    fmt = {c: "{:.4f}" for c in pct_cols}
    fmt.update({c: "{:,d}" for c in df_.columns if c not in pct_cols and df_[c].dtype.kind in "iu"})
    try:
        return df_.style.format(fmt, na_rep="n/a").background_gradient(
            cmap="Blues", vmin=0, vmax=1, subset=[c for c in pct_cols if c in df_.columns])
    except Exception:
        return df_.round(4)


print("self-test:", {k: round(v, 3) for k, v in
                     compute_metrics([0, 0, 1, 1], [.1, .4, .35, .8]).items()
                     if k in ("auroc", "f1", "sensitivity")})
print("self-test threshold_at_sensitivity:",
      round(threshold_at_sensitivity(np.array([0, 0, 1, 1, 1]),
                                     np.array([.1, .3, .4, .6, .9]), 0.9), 3))

## 11. Kiến trúc mạng nơ-ron 1D (8 của v2 + 2 mới ở v3)

In [ ]:
import torch.nn as nn


# ---------------------------------------------------------------- 1. PlainCNN
class PlainCNN(nn.Module):
    """Mốc sàn: Conv-BN-ReLU-MaxPool xếp chồng, không có gì đặc biệt."""

    def __init__(self, channels=(32, 64, 128, 256)):
        super().__init__()
        layers, c_in = [], NUM_LEADS
        for c_out in channels:
            layers += [nn.Conv1d(c_in, c_out, 7, padding=3, bias=False),
                       nn.BatchNorm1d(c_out), nn.ReLU(inplace=True), nn.MaxPool1d(4)]
            c_in = c_out
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Dropout(0.3), nn.Linear(c_in, 1))

    def forward(self, x):
        return self.head(self.features(x)).squeeze(-1)


# ---------------------------------------------------------------- 2. ResNet1D
class ResidualBlock1D(nn.Module):
    def __init__(self, c_in, c_out, k=7, stride=2, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(c_in, c_out, k, stride, k // 2, bias=False)
        self.bn1 = nn.BatchNorm1d(c_out)
        self.conv2 = nn.Conv1d(c_out, c_out, k, 1, k // 2, bias=False)
        self.bn2 = nn.BatchNorm1d(c_out)
        self.drop = nn.Dropout(dropout)
        self.relu = nn.ReLU(inplace=True)
        self.short = (nn.Identity() if (stride == 1 and c_in == c_out)
                      else nn.Sequential(nn.Conv1d(c_in, c_out, 1, stride, bias=False),
                                         nn.BatchNorm1d(c_out)))

    def forward(self, x):
        idt = self.short(x)
        out = self.drop(self.relu(self.bn1(self.conv1(x))))
        return self.relu(self.bn2(self.conv2(out)) + idt)


class ResNet1D(nn.Module):
    """Skip connection giúp gradient đi xuyên qua mạng sâu."""

    def __init__(self, channels=(32, 64, 128, 256), dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(NUM_LEADS, 32, 15, 2, 7, bias=False),
                                  nn.BatchNorm1d(32), nn.ReLU(inplace=True), nn.MaxPool1d(2))
        blocks, c_in = [], 32
        for c_out in channels:
            blocks.append(ResidualBlock1D(c_in, c_out, stride=2))
            c_in = c_out
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Linear(c_in, 64), nn.ReLU(inplace=True),
                                  nn.Dropout(dropout), nn.Linear(64, 1))

    def forward(self, x):
        return self.head(self.blocks(self.stem(x))).squeeze(-1)


# ---------------------------------------------------------------- 3. InceptionTime1D
class InceptionModule(nn.Module):
    """Nhiều độ dài kernel song song -> bắt được cả sóng nhanh lẫn biến thiên chậm."""

    def __init__(self, c_in, n_filters=32, kernels=(39, 19, 9), bottleneck=32):
        super().__init__()
        self.bottleneck = nn.Conv1d(c_in, bottleneck, 1, bias=False)
        self.convs = nn.ModuleList(
            [nn.Conv1d(bottleneck, n_filters, k, padding=k // 2, bias=False) for k in kernels])
        self.pool_conv = nn.Sequential(nn.MaxPool1d(3, stride=1, padding=1),
                                       nn.Conv1d(c_in, n_filters, 1, bias=False))
        self.bn = nn.BatchNorm1d(n_filters * (len(kernels) + 1))
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        b = self.bottleneck(x)
        return self.relu(self.bn(torch.cat([c(b) for c in self.convs] + [self.pool_conv(x)], 1)))


class InceptionTime1D(nn.Module):
    def __init__(self, n_filters=32, depth=6):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(NUM_LEADS, 32, 15, 4, 7, bias=False),
                                  nn.BatchNorm1d(32), nn.ReLU(inplace=True))
        c_out = n_filters * 4
        self.blocks = nn.ModuleList()
        self.shortcuts = nn.ModuleList()
        self.pools = nn.ModuleList()
        c_in = 32
        res_c = 32
        for d in range(depth):
            self.blocks.append(InceptionModule(c_in, n_filters))
            if d % 3 == 2:
                self.shortcuts.append(nn.Sequential(nn.Conv1d(res_c, c_out, 1, bias=False),
                                                    nn.BatchNorm1d(c_out)))
                self.pools.append(nn.MaxPool1d(4))
                res_c = c_out
            else:
                self.shortcuts.append(None)
                self.pools.append(None)
            c_in = c_out
        self.relu = nn.ReLU(inplace=True)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Dropout(0.3), nn.Linear(c_out, 1))

    def forward(self, x):
        x = self.stem(x)
        res = x
        for blk, short, pool in zip(self.blocks, self.shortcuts, self.pools):
            x = blk(x)
            if short is not None:
                x = self.relu(x + short(res))
                x = pool(x)
                res = x
        return self.head(x).squeeze(-1)


# ---------------------------------------------------------------- 4. CNN + BiLSTM
class CNNBiLSTM(nn.Module):
    """CNN rút đặc trưng cục bộ, BiLSTM mô hình hoá quan hệ theo thời gian giữa các nhịp."""

    def __init__(self, hidden=128):
        super().__init__()
        layers, c_in = [], NUM_LEADS
        for c_out in (32, 64, 128):
            layers += [nn.Conv1d(c_in, c_out, 7, padding=3, bias=False),
                       nn.BatchNorm1d(c_out), nn.ReLU(inplace=True), nn.MaxPool1d(4)]
            c_in = c_out
        self.cnn = nn.Sequential(*layers)
        self.lstm = nn.LSTM(c_in, hidden, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(hidden * 2, 1))

    def forward(self, x):
        h = self.cnn(x).transpose(1, 2)
        out, _ = self.lstm(h)
        return self.head(out.mean(dim=1)).squeeze(-1)


# ---------------------------------------------------------------- 5. SEResNet1D
class SEBlock1D(nn.Module):
    """Squeeze-excitation: học trọng số quan trọng theo từng kênh (feature map)."""

    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(nn.Linear(channels, hidden), nn.ReLU(inplace=True),
                                nn.Linear(hidden, channels), nn.Sigmoid())

    def forward(self, x):
        w = self.fc(self.pool(x).squeeze(-1)).unsqueeze(-1)
        return x * w


class SEResidualBlock1D(nn.Module):
    def __init__(self, c_in, c_out, k=7, stride=2, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(c_in, c_out, k, stride, k // 2, bias=False)
        self.bn1 = nn.BatchNorm1d(c_out)
        self.conv2 = nn.Conv1d(c_out, c_out, k, 1, k // 2, bias=False)
        self.bn2 = nn.BatchNorm1d(c_out)
        self.se = SEBlock1D(c_out)
        self.drop = nn.Dropout(dropout)
        self.relu = nn.ReLU(inplace=True)
        self.short = (nn.Identity() if (stride == 1 and c_in == c_out)
                      else nn.Sequential(nn.Conv1d(c_in, c_out, 1, stride, bias=False),
                                         nn.BatchNorm1d(c_out)))

    def forward(self, x):
        idt = self.short(x)
        out = self.drop(self.relu(self.bn1(self.conv1(x))))
        out = self.se(self.bn2(self.conv2(out)))
        return self.relu(out + idt)


class SEResNet1D(nn.Module):
    """ResNet1D + squeeze-excitation sau mỗi khối."""

    def __init__(self, channels=(32, 64, 128, 256), dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(NUM_LEADS, 32, 15, 2, 7, bias=False),
                                  nn.BatchNorm1d(32), nn.ReLU(inplace=True), nn.MaxPool1d(2))
        blocks, c_in = [], 32
        for c_out in channels:
            blocks.append(SEResidualBlock1D(c_in, c_out, stride=2))
            c_in = c_out
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Linear(c_in, 64), nn.ReLU(inplace=True),
                                  nn.Dropout(dropout), nn.Linear(64, 1))

    def forward(self, x):
        return self.head(self.blocks(self.stem(x))).squeeze(-1)


# ---------------------------------------------------------------- 6. XResNet1D
class XResBlock1D(nn.Module):
    """Nhánh tắt kiểu xResNet: AvgPool + 1x1 conv thay vì strided conv khi downsample."""

    def __init__(self, c_in, c_out, k=7, stride=2, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(c_in, c_out, k, stride, k // 2, bias=False)
        self.bn1 = nn.BatchNorm1d(c_out)
        self.conv2 = nn.Conv1d(c_out, c_out, k, 1, k // 2, bias=False)
        self.bn2 = nn.BatchNorm1d(c_out)
        self.drop = nn.Dropout(dropout)
        self.act = nn.SiLU(inplace=True)
        if stride == 1 and c_in == c_out:
            self.short = nn.Identity()
        else:
            self.short = nn.Sequential(
                nn.AvgPool1d(stride, ceil_mode=True), nn.Conv1d(c_in, c_out, 1, bias=False),
                nn.BatchNorm1d(c_out))

    def forward(self, x):
        idt = self.short(x)
        out = self.drop(self.act(self.bn1(self.conv1(x))))
        out = self.bn2(self.conv2(out))
        return self.act(out + idt)


class XResNet1D(nn.Module):
    """Stem sâu (3 conv nhỏ xếp chồng) thay vì 1 conv to + SiLU -- biến thể tối ưu hoá của ResNet1D."""

    def __init__(self, channels=(32, 64, 128, 256), dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(NUM_LEADS, 32, 5, 2, 2, bias=False), nn.BatchNorm1d(32), nn.SiLU(inplace=True),
            nn.Conv1d(32, 32, 5, 1, 2, bias=False), nn.BatchNorm1d(32), nn.SiLU(inplace=True),
            nn.Conv1d(32, 32, 5, 1, 2, bias=False), nn.BatchNorm1d(32), nn.SiLU(inplace=True),
            nn.MaxPool1d(2))
        blocks, c_in = [], 32
        for c_out in channels:
            blocks.append(XResBlock1D(c_in, c_out, stride=2))
            c_in = c_out
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Linear(c_in, 64), nn.SiLU(inplace=True),
                                  nn.Dropout(dropout), nn.Linear(64, 1))

    def forward(self, x):
        return self.head(self.blocks(self.stem(x))).squeeze(-1)


# ---------------------------------------------------------------- 7. ConvNeXtV2_1D
class GRN1D(nn.Module):
    """Global Response Normalization (ConvNeXt V2) -- chuẩn hoá theo norm toàn cục của từng kênh."""

    def __init__(self, channels):
        super().__init__()
        self.gamma = nn.Parameter(torch.zeros(1, 1, channels))
        self.beta = nn.Parameter(torch.zeros(1, 1, channels))

    def forward(self, x):
        gx = torch.norm(x, p=2, dim=1, keepdim=True)
        nx = gx / (gx.mean(dim=-1, keepdim=True) + 1e-6)
        return self.gamma * (x * nx) + self.beta + x


class ConvNeXtV2Block1D(nn.Module):
    def __init__(self, channels, expand=4, dropout=0.1):
        super().__init__()
        self.dwconv = nn.Conv1d(channels, channels, 7, padding=3, groups=channels, bias=False)
        self.norm = nn.LayerNorm(channels)
        self.pw1 = nn.Linear(channels, channels * expand)
        self.act = nn.GELU()
        self.grn = GRN1D(channels * expand)
        self.pw2 = nn.Linear(channels * expand, channels)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        idt = x
        x = self.dwconv(x).transpose(1, 2)
        x = self.norm(x)
        x = self.pw2(self.grn(self.act(self.pw1(x))))
        x = self.drop(x).transpose(1, 2)
        return x + idt


class ConvNeXtV2Downsample1D(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.norm = nn.BatchNorm1d(c_in)
        self.conv = nn.Conv1d(c_in, c_out, 2, stride=2, bias=False)

    def forward(self, x):
        return self.conv(self.norm(x))


class ConvNeXtV2_1D(nn.Module):
    """Khối depthwise-conv + LayerNorm + MLP mở rộng + GRN -- bản 1D của ConvNeXt V2."""

    def __init__(self, channels=(32, 64, 128, 256), depths=(1, 1, 2, 1), dropout=0.1):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(NUM_LEADS, channels[0], 4, stride=4, bias=False),
                                  nn.BatchNorm1d(channels[0]))
        stages, c_in = [], channels[0]
        for stage_i, (c_out, depth) in enumerate(zip(channels, depths)):
            if stage_i > 0:
                stages.append(ConvNeXtV2Downsample1D(c_in, c_out))
            stages += [ConvNeXtV2Block1D(c_out, dropout=dropout) for _ in range(depth)]
            c_in = c_out
        self.stages = nn.Sequential(*stages)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.LayerNorm(c_in), nn.Dropout(dropout), nn.Linear(c_in, 1))

    def forward(self, x):
        return self.head(self.stages(self.stem(x))).squeeze(-1)


# ---------------------------------------------------------------- 8. AiTiAMI (tái hiện)
class AiTiAMIBlock1D(nn.Module):
    """Residual block chuẩn, kernel hẹp hơn ResNet1D (5 thay vì 7) để bắt chi tiết QRS/ST mịn hơn."""

    def __init__(self, c_in, c_out, k=5, stride=2, dropout=0.15):
        super().__init__()
        self.conv1 = nn.Conv1d(c_in, c_out, k, stride, k // 2, bias=False)
        self.bn1 = nn.BatchNorm1d(c_out)
        self.conv2 = nn.Conv1d(c_out, c_out, k, 1, k // 2, bias=False)
        self.bn2 = nn.BatchNorm1d(c_out)
        self.drop = nn.Dropout(dropout)
        self.relu = nn.ReLU(inplace=True)
        self.short = (nn.Identity() if (stride == 1 and c_in == c_out)
                      else nn.Sequential(nn.Conv1d(c_in, c_out, 1, stride, bias=False),
                                         nn.BatchNorm1d(c_out)))

    def forward(self, x):
        idt = self.short(x)
        out = self.drop(self.relu(self.bn1(self.conv1(x))))
        return self.relu(self.bn2(self.conv2(out)) + idt)


class AiTiAMI(nn.Module):
    """Tái hiện theo MÔ TẢ KIẾN TRÚC trong Lee et al., Eur Heart J 2025 (ehaf004, ROMIAE):
    "an advanced algorithm... built on a residual neural network", đầu vào 500Hz 12 chuyển đạo.
    KHÔNG PHẢI model/trọng số gốc của Medical AI Co., Ltd -- bài báo không công bố kiến trúc chi
    tiết (sản phẩm thương mại). Sâu hơn (5 stage) và rộng hơn ResNet1D, head avg+max-pool nối tiếp.
    """

    def __init__(self, channels=(48, 96, 192, 256, 320), dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(NUM_LEADS, 48, 15, 2, 7, bias=False),
                                  nn.BatchNorm1d(48), nn.ReLU(inplace=True), nn.MaxPool1d(2))
        blocks, c_in = [], 48
        for i, c_out in enumerate(channels):
            blocks.append(AiTiAMIBlock1D(c_in, c_out, stride=2 if i < 4 else 1))
            c_in = c_out
        self.blocks = nn.Sequential(*blocks)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.maxpool = nn.AdaptiveMaxPool1d(1)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(c_in * 2, 128), nn.ReLU(inplace=True),
                                  nn.Dropout(dropout), nn.Linear(128, 1))

    def forward(self, x):
        feat = self.blocks(self.stem(x))
        pooled = torch.cat([self.avgpool(feat), self.maxpool(feat)], dim=1)
        return self.head(pooled).squeeze(-1)


# ---------------------------------------------------------------- 9. TCN1D  [MỚI ở v3]
class TCNBlock1D(nn.Module):
    """Khối residual dùng convolution GIÃN NỞ (dilated): với cùng số tham số, trường thu nhận
    rộng gấp `dilation` lần so với conv thường. padding = (k-1)*d//2 với k lẻ giữ nguyên độ dài."""

    def __init__(self, c_in, c_out, k=7, dilation=1, dropout=0.1):
        super().__init__()
        pad = (k - 1) * dilation // 2
        self.conv1 = nn.Conv1d(c_in, c_out, k, padding=pad, dilation=dilation, bias=False)
        self.bn1 = nn.BatchNorm1d(c_out)
        self.conv2 = nn.Conv1d(c_out, c_out, k, padding=pad, dilation=dilation, bias=False)
        self.bn2 = nn.BatchNorm1d(c_out)
        self.drop = nn.Dropout(dropout)
        self.act = nn.ReLU(inplace=True)
        self.short = nn.Identity() if c_in == c_out else nn.Conv1d(c_in, c_out, 1, bias=False)

    def forward(self, x):
        idt = self.short(x)
        out = self.drop(self.act(self.bn1(self.conv1(x))))
        return self.act(self.bn2(self.conv2(out)) + idt)


class TCN1D(nn.Module):
    """Temporal Convolutional Network: dilation 1-2-4-8 cho trường thu nhận tăng theo cấp số
    nhân, phủ trọn nhiều nhịp liên tiếp mà không cần pooling sâu như nhóm ResNet.

    Lý do đưa vào v3: 4/8 kiến trúc của v2 (ResNet1D, SEResNet1D, XResNet1D, AiTiAMI) đều là
    residual-CNN kernel 5-7, lỗi tương quan cao nên ensemble gần như không lợi thêm. TCN có
    inductive bias khác hẳn -> sai ở những ca khác -> đó mới là thứ ensemble khai thác được.
    """

    def __init__(self, channels=(48, 64, 96, 128), dilations=(1, 2, 4, 8), dropout=0.2):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(NUM_LEADS, 48, 15, 4, 7, bias=False),
                                  nn.BatchNorm1d(48), nn.ReLU(inplace=True), nn.MaxPool1d(2))
        blocks, c_in = [], 48
        for c_out, d in zip(channels, dilations):
            blocks += [TCNBlock1D(c_in, c_out, dilation=d, dropout=dropout), nn.MaxPool1d(2)]
            c_in = c_out
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Dropout(0.3), nn.Linear(c_in, 1))

    def forward(self, x):
        return self.head(self.blocks(self.stem(x))).squeeze(-1)


# ------------------------------------------------------- 10. CNNTransformer  [MỚI ở v3]
class CNNTransformer(nn.Module):
    """CNN rút đặc trưng cục bộ -> Transformer encoder tự chú ý giữa các đoạn thời gian.

    Khác CNN+BiLSTM ở chỗ: BiLSTM đọc tuần tự và thông tin xa bị suy giảm qua từng bước, còn
    self-attention nối trực tiếp mọi cặp vị trí. Với STEMI -- nơi phải đối chiếu mức chênh ST
    ở đoạn này với đường đẳng điện ở đoạn khác -- đây là bias hợp lý và khác biệt.
    norm_first=True (pre-LN) để ổn định khi train từ đầu, không cần lịch LR phức tạp.
    """

    def __init__(self, d_model=128, nhead=4, layers=2, dropout=0.1):
        super().__init__()
        conv, c_in = [], NUM_LEADS
        for c_out in (32, 64, d_model):
            conv += [nn.Conv1d(c_in, c_out, 7, padding=3, bias=False),
                     nn.BatchNorm1d(c_out), nn.ReLU(inplace=True), nn.MaxPool1d(4)]
            c_in = c_out
        self.cnn = nn.Sequential(*conv)                     # 5000 -> 1250 -> 312 -> 78
        self.pos = nn.Parameter(torch.zeros(1, SIGNAL_LEN // 64 + 2, d_model))
        enc_layer = nn.TransformerEncoderLayer(
            d_model, nhead, dim_feedforward=d_model * 4, dropout=dropout,
            batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(d_model, 1))

    def forward(self, x):
        h = self.cnn(x).transpose(1, 2)                     # (B, T, C)
        h = h + self.pos[:, :h.size(1)]
        h = self.norm(self.encoder(h))
        return self.head(h.mean(dim=1)).squeeze(-1)


MODELS = {
    "PlainCNN": PlainCNN, "ResNet1D": ResNet1D, "InceptionTime1D": InceptionTime1D,
    "CNN+BiLSTM": CNNBiLSTM, "SEResNet1D": SEResNet1D, "XResNet1D": XResNet1D,
    "ConvNeXtV2_1D": ConvNeXtV2_1D, "AiTiAMI": AiTiAMI,
    "TCN1D": TCN1D, "CNNTransformer": CNNTransformer,
}
assert set(CANDIDATES) <= set(MODELS), \
    f"CANDIDATES có tên không nằm trong MODELS: {sorted(set(CANDIDATES) - set(MODELS))}"

rows = []
for name in CANDIDATES:
    m = MODELS[name]().to(DEVICE)
    with torch.no_grad():
        out = m(torch.zeros(2, NUM_LEADS, SIGNAL_LEN, device=DEVICE))
    assert tuple(out.shape) == (2,), f"{name} trả về shape {tuple(out.shape)}, cần (B,)"
    rows.append({"Mô hình": name, "Tham số": sum(p.numel() for p in m.parameters()),
                 "Output": str(tuple(out.shape)),
                 "LR": MODEL_LR_OVERRIDE.get(name, LR)})
    del m
if GPU_AVAILABLE:
    torch.cuda.empty_cache()
display(pd.DataFrame(rows).set_index("Mô hình"))

## 12. Hàm loss: Focal Loss + label smoothing

`BCEWithLogitsLoss` không có tham số `label_smoothing` sẵn (khác `CrossEntropyLoss`), nên làm
mềm target thủ công: `y_smooth = y*(1-eps) + eps/2`. Hai cơ chế được **tách riêng**: hệ số điều
chỉnh focal `(1-p_t)^γ` và trọng số lớp `α` dùng **nhãn cứng gốc** (đo đúng model đang tự tin
hay không với đúng lớp thật), còn target **đã làm mềm** chỉ dùng khi tính giá trị BCE — trộn
lẫn hai việc này sẽ làm sai ý nghĩa của cả hai cơ chế.

`α` (trọng số dương/âm) suy ra từ **đúng tỷ lệ lớp của tập train mỗi fold** — α mặc định 0,25
của bài báo RetinaNet giả định mất cân bằng ~1:1000, khác xa tỷ lệ dương ~6-8% (~1:12) ở đây.

In [ ]:
class FocalLossWithSmoothing(nn.Module):
    """BCE hai lớp có trọng số alpha (theo tỷ lệ lớp thật) + điều chỉnh focal gamma (tập trung
    vào ca khó) + label smoothing (giảm tự tin quá mức). gamma=0 -> quay lại BCE có trọng số
    lớp thuần tuý; label_smoothing=0 -> tắt làm mềm target.
    """

    def __init__(self, alpha_pos: float, gamma: float = 2.0, label_smoothing: float = 0.0):
        super().__init__()
        assert 0.0 < alpha_pos < 1.0
        self.alpha_pos = float(alpha_pos)
        self.gamma = float(gamma)
        self.eps = float(label_smoothing)

    def forward(self, logits, targets):
        targets = targets.float()
        targets_smooth = targets * (1 - self.eps) + self.eps / 2 if self.eps > 0 else targets

        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets_smooth, reduction="none")
        if self.gamma <= 0 and self.alpha_pos == 0.5:
            return bce.mean()

        with torch.no_grad():
            p = torch.sigmoid(logits)
            p_t = p * targets + (1 - p) * (1 - targets)          # xác suất gán cho nhãn CỨNG thật
            alpha_t = self.alpha_pos * targets + (1 - self.alpha_pos) * (1 - targets)
            focal_w = alpha_t * (1 - p_t).clamp(min=1e-6, max=1.0) ** self.gamma
        return (focal_w * bce).mean()


# self-test: loss phải giảm khi logit dự đoán đúng hướng nhãn, và focal phải hạ trọng số ca dễ
_crit = FocalLossWithSmoothing(alpha_pos=0.5, gamma=2.0, label_smoothing=0.02)
_logit_easy = torch.tensor([5.0])    # rất tự tin, đúng
_logit_hard = torch.tensor([0.0])    # không chắc chắn
_target = torch.tensor([1.0])
_l_easy = _crit(_logit_easy, _target).item()
_l_hard = _crit(_logit_hard, _target).item()
_verdict = "OK" if _l_easy < _l_hard else "LOI"
print(f"self-test FocalLossWithSmoothing: loss(ca dễ)={_l_easy:.4f} < loss(ca khó)={_l_hard:.4f} -> {_verdict}")

## 13. Huấn luyện K-Fold, thu thập dự đoán out-of-fold (OOF)

Mỗi kiến trúc huấn luyện `K_FOLDS` lần (một lần/fold); dự đoán trên tập val của đúng fold đó
được ghép lại thành **OOF** — đúng cho mọi bản ghi trong POOL đúng 1 dự đoán, không rò rỉ.
Checkpoint được khoá theo *fingerprint* (kiến trúc, fold, seed, augmentation, tham số loss...):
đổi bất kỳ tham số nào trong đó sẽ tự động huấn luyện lại thay vì âm thầm dùng checkpoint cũ.

In [ ]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

OOF_DIR.mkdir(parents=True, exist_ok=True)
OOF_NPZ = OOF_DIR / f"oof_{TARGET_LABEL}_{RUN_MODE}_k{K_FOLDS}_{EXPERIMENT_TAG}.npz"
OOF_JSON = OOF_NPZ.with_suffix(".meta.json")

# Fingerprint phải chứa MỌI siêu tham số ảnh hưởng tới trọng số. Thiếu một cái là lần chạy sau
# sẽ âm thầm nạp lại checkpoint cũ (train bằng cấu hình khác) và kết quả không còn giải thích
# được. v3 bổ sung warmup + lr_override so với v2.
CV_FP = dict(run_mode=RUN_MODE, target=TARGET_LABEL, k=K_FOLDS, seed=SEED, epochs=EPOCHS,
             n_pool=len(POOL_IDX), models=CANDIDATES, n_folds_run=N_FOLDS_RUN,
             preprocess_version=PREPROCESS_VERSION, aug_enable=AUG_ENABLE,
             aug_max_shift=AUG_MAX_SHIFT, aug_noise_std=AUG_NOISE_STD,
             aug_gain_jitter=AUG_GAIN_JITTER, focal_gamma=FOCAL_GAMMA,
             label_smooth_eps=LABEL_SMOOTH_EPS, weight_decay=WEIGHT_DECAY,
             warmup_epochs=WARMUP_EPOCHS, lr=LR,
             lr_override=dict(sorted(MODEL_LR_OVERRIDE.items())))

GRAD_CLIP_NORM = 1.0

NAN = float("nan")
OOF_P = {m: np.full(len(df), NAN) for m in CANDIDATES}
FOLD_ID = np.full(len(df), -1, dtype=int)
RUN_INFO = {}


def evaluate(model, loader):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                logits = model(xb)
            ys.append(yb.numpy())
            ps.append(torch.sigmoid(logits.float()).cpu().numpy())
    y_out, p_out = np.concatenate(ys), np.concatenate(ps)
    bad = ~np.isfinite(p_out)
    if bad.any():
        print(f"  !! {bad.sum()} xác suất không hữu hạn -> thay bằng 0.5; model đang bất ổn định.")
        p_out = np.where(bad, 0.5, p_out)
    return y_out, p_out


def _apply_warmup_lr(optimizer, base_lr, epoch):
    """[v3] Warmup tuyến tính: epoch 0..WARMUP_EPOCHS-1 chạy LR = base_lr*(epoch+1)/WARMUP.
    Trả về True nếu vẫn đang trong giai đoạn warmup (khi đó CHƯA cho scheduler can thiệp)."""
    if WARMUP_EPOCHS <= 0 or epoch >= WARMUP_EPOCHS:
        return False
    for g in optimizer.param_groups:
        g["lr"] = base_lr * (epoch + 1) / WARMUP_EPOCHS
    return epoch < WARMUP_EPOCHS - 1        # epoch cuối của warmup đã ở đúng base_lr


def train_fold(name, cls, k, tr_loader, va_loader, alpha_pos, tag):
    ckpt_path = MODEL_DIR / f'{name.replace("+", "_")}_f{k}_best.pt'
    fp = dict(CV_FP, model=name, fold=k)
    fp.pop("models"), fp.pop("n_folds_run")

    if ckpt_path.exists():
        ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        if ck.get("fp") == fp:
            model = cls().to(DEVICE)
            model.load_state_dict(ck["model"])
            print(f"  [{tag}] đã có checkpoint khớp -> bỏ qua train "
                  f"(best AUPRC {ck['best']:.4f} @ ep {ck['best_epoch']})")
            return model, ck["hist"], ck["best_epoch"], ck["seconds"]

    torch.manual_seed(SEED + k)
    np.random.seed(SEED + k)
    model = cls().to(DEVICE)
    base_lr = MODEL_LR_OVERRIDE.get(name, LR)
    criterion = FocalLossWithSmoothing(alpha_pos=alpha_pos, gamma=FOCAL_GAMMA,
                                       label_smoothing=LABEL_SMOOTH_EPS)
    optimizer = torch.optim.AdamW(model.parameters(), lr=base_lr, weight_decay=WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=.5, patience=LR_PATIENCE)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    best, best_epoch, bad, hist = -np.inf, -1, 0, []
    t_all = time.time()
    for epoch in range(EPOCHS):
        in_warmup = _apply_warmup_lr(optimizer, base_lr, epoch)
        model.train()
        tot, seen = 0.0, 0
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                loss = criterion(model(xb), yb)
            if not torch.isfinite(loss):
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            tot += loss.item() * xb.size(0)
            seen += xb.size(0)

        va_y, va_p = evaluate(model, va_loader)
        m = compute_metrics(va_y, va_p)
        if not in_warmup:                       # scheduler chỉ chạy sau khi warmup xong
            scheduler.step(m["auprc"] if not np.isnan(m["auprc"]) else -np.inf)
        hist.append({"epoch": epoch + 1, "train_loss": tot / max(seen, 1),
                     "val_auprc": m["auprc"], "val_auroc": m["auroc"],
                     "lr": optimizer.param_groups[0]["lr"]})

        if (not np.isnan(m["auprc"])) and m["auprc"] > best:
            best, best_epoch, bad = float(m["auprc"]), epoch + 1, 0
            torch.save({"model": model.state_dict(), "fp": fp, "best": best,
                        "best_epoch": best_epoch, "hist": hist,
                        "seconds": round(time.time() - t_all, 1)}, ckpt_path)
        else:
            bad += 1
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  [{tag}] ep {epoch + 1:>3}/{EPOCHS}  loss {tot / max(seen, 1):.4f}  "
                  f"val auprc {m['auprc']:.4f}  auroc {m['auroc']:.4f}  "
                  f"lr {optimizer.param_groups[0]['lr']:.2e}")
        if bad >= EARLY_STOP_PATIENCE:
            print(f"  [{tag}] early stop @ epoch {epoch + 1}")
            break

    seconds = round(time.time() - t_all, 1)
    ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    ck["seconds"], ck["hist"] = seconds, hist
    torch.save(ck, ckpt_path)
    model.load_state_dict(ck["model"])
    print(f"  [{tag}] xong {len(hist)} epoch trong {seconds:.0f}s | best AUPRC {best:.4f} @ ep {best_epoch}")
    return model, hist, best_epoch, seconds


def _oof_ready():
    if not (OOF_NPZ.exists() and OOF_JSON.exists()):
        return False
    if json.loads(OOF_JSON.read_text()).get("fp") != CV_FP:
        return False
    z = np.load(OOF_NPZ, allow_pickle=False)
    return all(f"p_{m}" in z for m in CANDIDATES)


if _oof_ready():
    _z = np.load(OOF_NPZ, allow_pickle=False)
    for m in CANDIDATES:
        OOF_P[m] = _z[f"p_{m}"]
    FOLD_ID = _z["fold_id"]
    RUN_INFO = json.loads(OOF_JSON.read_text())["runs"]
    print(f"Đã có OOF đầy đủ: {OOF_NPZ.name} -> bỏ qua toàn bộ phần huấn luyện.")
else:
    for k in range(N_FOLDS_RUN):
        tr_idx_k, va_idx_k = FOLDS[k]
        mean_k, std_k = norm_stats(tr_idx_k)
        tr_loader, va_loader = make_loaders(tr_idx_k, va_idx_k, mean_k, std_k)
        n_pos_k = int(y[tr_idx_k].sum())
        alpha_pos_k = 1.0 - n_pos_k / len(tr_idx_k)     # tỷ lệ lớp ÂM trong train -> trọng số cho lớp DƯƠNG
        FOLD_ID[va_idx_k] = k
        print(f"\n=== fold {k} " + "=" * 55 +
              f"\n    train {len(tr_idx_k):,} ({n_pos_k} dương, alpha_pos {alpha_pos_k:.3f}) | "
              f"val {len(va_idx_k):,} ({int(y[va_idx_k].sum())} dương)")
        for name in CANDIDATES:
            tag = f"{name} f{k}"
            model, hist, best_epoch, secs = train_fold(
                name, MODELS[name], k, tr_loader, va_loader, alpha_pos_k, tag)
            _, p = evaluate(model, va_loader)
            OOF_P[name][va_idx_k] = p
            RUN_INFO[f"{name}|{k}"] = {
                "best_epoch": best_epoch, "seconds": secs, "n_epoch": len(hist),
                "params": sum(q.numel() for q in model.parameters()), "hist": hist}
            del model
            if GPU_AVAILABLE:
                torch.cuda.empty_cache()

    np.savez_compressed(OOF_NPZ, y=y, fold_id=FOLD_ID,
                        patient=df[COL_PATIENT].astype(str).to_numpy(),
                        **{f"p_{m}": OOF_P[m] for m in CANDIDATES})
    OOF_JSON.write_text(json.dumps({"fp": CV_FP, "runs": RUN_INFO}, ensure_ascii=False))
    print(f"\nĐã ghi OOF -> {OOF_NPZ}")

VALID = FOLD_ID >= 0
print(f"OOF phủ {VALID.sum():,}/{len(df):,} bản ghi | {int(y[VALID].sum())} ca dương")

## 14. Kết quả từng fold và độ ổn định

Kiểm tra nhanh: chênh lệch AUPRC/AUROC giữa các fold của cùng một model phản ánh phương sai
tự nhiên của việc chia dữ liệu — nếu độ lệch chuẩn giữa các fold còn lớn hơn khoảng cách xếp
hạng giữa hai model, thứ hạng đó chưa có căn cứ để kết luận.

In [ ]:
def youden_threshold(y_true, y_prob):
    if len(np.unique(np.asarray(y_true).astype(int))) < 2:
        return 0.5
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    return float(np.clip(thr[int(np.argmax(tpr - fpr))], 0, 1))


rows = []
for name in CANDIDATES:
    for k in range(N_FOLDS_RUN):
        m_fold = FOLD_ID == k
        if not m_fold.any():
            continue
        yk, pk = y[m_fold], OOF_P[name][m_fold]
        met = compute_metrics(yk, pk, youden_threshold(yk, pk))
        rows.append({"Model": name, "fold": k, "AUROC": met["auroc"], "AUPRC": met["auprc"],
                     "Sensitivity": met["sensitivity"], "Specificity": met["specificity"],
                     "F1": met["f1"], "n": met["n"], "n_pos": met["n_pos"]})

FOLD_METRICS = pd.DataFrame(rows)
PCT_CV = ["AUROC", "AUPRC", "Sensitivity", "Specificity", "F1"]
display(style_df(FOLD_METRICS.set_index(["Model", "fold"]), PCT_CV))

summary_cv = FOLD_METRICS.groupby("Model")[["AUROC", "AUPRC", "Sensitivity", "Specificity", "F1"]].agg(["mean", "std"])
summary_cv.columns = [f"{c}_{s}" for c, s in summary_cv.columns]
summary_cv = summary_cv.sort_values("AUPRC_mean", ascending=False)
print("\nTrung bình ± độ lệch chuẩn qua các fold (xếp theo AUPRC giảm dần):")
display(style_df(summary_cv, [c for c in summary_cv.columns if c.endswith("_mean")]))

## 14b. Chẩn đoán trọng số Focal Loss (mới, không cần train lại)

`alpha_pos` đã tự điều chỉnh theo đúng tỷ lệ lớp của từng fold (~8% dương). Câu hỏi cần trả
lời bằng số liệu: phần điều biến `gamma=FOCAL_GAMMA` — vốn thiết kế cho mất cân bằng cực đoan
kiểu object detection (~1:1000) — có đang hạ trọng số ca dương "dễ" MẠNH HƠN ca âm hay không,
dù alpha đã ưu tiên lớp dương? Nếu có, đó là dấu hiệu gamma đang phản tác dụng thay vì hỗ trợ.
Dùng lại chính xác suất OOF đã có (không tốn thêm GPU).

In [ ]:
def focal_weight_stats(y_true, y_prob, alpha_pos, gamma, eps=1e-6):
    """Trọng số focal trung bình cho ca dương/âm, dùng lại đúng công thức trong FocalLossWithSmoothing."""
    y_true = np.asarray(y_true).astype(int)
    p = np.clip(np.asarray(y_prob, dtype=np.float64), eps, 1 - eps)
    p_t = np.where(y_true == 1, p, 1 - p)
    alpha_t = np.where(y_true == 1, alpha_pos, 1 - alpha_pos)
    w = alpha_t * np.clip(1 - p_t, eps, 1.0) ** gamma
    return float(w[y_true == 1].mean()), float(w[y_true == 0].mean())


# alpha_pos thực tế của từng fold -- tính lại trực tiếp từ FOLDS, không phụ thuộc việc OOF được
# train mới hay load từ cache (luôn đúng trong cả hai trường hợp)
FOLD_ALPHA = {k: 1.0 - float(y[FOLDS[k][0]].sum()) / len(FOLDS[k][0])
              for k in range(N_FOLDS_RUN) if k in FOLDS}

rows = []
for name in CANDIDATES:
    pos_w, neg_w = [], []
    for k in range(N_FOLDS_RUN):
        m_fold = FOLD_ID == k
        if not m_fold.any() or k not in FOLD_ALPHA:
            continue
        wp, wn = focal_weight_stats(y[m_fold], OOF_P[name][m_fold], FOLD_ALPHA[k], FOCAL_GAMMA)
        pos_w.append(wp)
        neg_w.append(wn)
    rows.append({"Model": name, "TB trong so ca duong": np.mean(pos_w),
                 "TB trong so ca am": np.mean(neg_w),
                 "Ty le duong/am": np.mean(pos_w) / np.mean(neg_w)})

focal_diag_df = pd.DataFrame(rows).set_index("Model").sort_values("Ty le duong/am")
print(f"gamma={FOCAL_GAMMA} | alpha trung bình các fold ~ {np.mean(list(FOLD_ALPHA.values())):.3f}")
display(focal_diag_df.style.format("{:.4f}"))

_ratio_low = (focal_diag_df["Ty le duong/am"] < 0.8).sum()
if _ratio_low >= len(focal_diag_df) // 2:
    print(f"\n!! {_ratio_low}/{len(focal_diag_df)} model có tỷ lệ trọng số dương/âm < 0.8 --")
    print("   gamma đang hạ trọng số ca dương mạnh hơn ca âm dù alpha đã ưu tiên lớp dương.")
    print("   Khuyến nghị: thử lại với FOCAL_GAMMA thấp hơn (vd. 1.0 hoặc 0 = BCE có trọng số")
    print("   thuần) và so sánh AUPRC/ECE trên POOL (mục 16) trước khi chốt gamma cho báo cáo.")
else:
    print("\nTrọng số dương/âm không lệch quá mạnh -- gamma hiện tại không có dấu hiệu")
    print("hạ trọng số ca dương quá mức so với những gì alpha đã tự điều chỉnh.")


## 15. Xem trước ngưỡng Sensitivity ≥ 91% của từng model đơn trên POOL

Chỉ là bảng tham khảo cho model đơn. Ngưỡng **thật sự dùng để báo cáo** — có biên an toàn và
tính cho cả các ensemble — được chốt ở mục 19.

In [ ]:
POOL_THRESHOLDS = {}
rows = []
for name in CANDIDATES:
    p_ = OOF_P[name][VALID]
    t = threshold_at_sensitivity(y[VALID], p_, TARGET_SENSITIVITY)
    POOL_THRESHOLDS[name] = t
    m = compute_metrics(y[VALID], p_, t)
    rows.append({"Model": name, "Ngưỡng": t, "AUROC": m["auroc"], "AUPRC": m["auprc"],
                 "Sensitivity": m["sensitivity"], "Specificity": m["specificity"],
                 "PPV": m["ppv"], "NPV": m["npv"], "F1": m["f1"]})

pool_thr_df = pd.DataFrame(rows).set_index("Model").sort_values("AUPRC", ascending=False)
print(f"POOL (OOF, {int(VALID.sum()):,} bản ghi) tại Sensitivity >= {TARGET_SENSITIVITY:.0%}:")
display(style_df(pool_thr_df, ["Ngưỡng", "AUROC", "AUPRC", "Sensitivity", "Specificity", "PPV", "NPV", "F1"]))

## 16. Hiệu chuẩn + vườn ensemble — chọn và KHOÁ quán quân trên OOF

Đây là mục quyết định của v3. Sáu cách tổ hợp được dựng và chấm điểm **chỉ trên OOF của POOL**;
các phương án có tham số (Greedy, Top-K, Stacking) được **cross-fit** để điểm số trung thực.
Quán quân khoá lại ở cuối mục này — từ mục 18 trở đi chỉ áp dụng, không chọn lại.

In [ ]:
from sklearn.linear_model import LogisticRegression

# =============================================================================
# NGUYÊN TẮC CỦA CẢ MỤC NÀY: mọi thứ chỉ nhìn OOF (POOL). Quán quân được CHỌN và
# KHOÁ ở đây, trước khi bất kỳ dòng nào chạm vào TEST.
# =============================================================================


def _logit(p, eps=1e-6):
    p = np.clip(np.asarray(p, dtype=np.float64), eps, 1 - eps)
    return np.log(p / (1 - p))


def crossfit_platt(p_raw, fold_id, valid_mask):
    """Fold k được hiệu chuẩn bằng bộ Platt fit trên OOF của các fold KHÁC."""
    out = np.full(len(p_raw), NAN)
    folds_present = [k for k in range(K_FOLDS) if (fold_id == k).any()]
    for k in folds_present:
        fit_mask = (fold_id != k) & valid_mask
        apply_mask = fold_id == k
        if fit_mask.sum() == 0 or len(np.unique(y[fit_mask].astype(int))) < 2:
            out[apply_mask] = p_raw[apply_mask]          # debug 1 fold -> không cross-fit được
            continue
        lr = LogisticRegression(C=1e6, solver="lbfgs", max_iter=1000)
        lr.fit(_logit(p_raw[fit_mask]).reshape(-1, 1), y[fit_mask].astype(int))
        out[apply_mask] = lr.predict_proba(_logit(p_raw[apply_mask]).reshape(-1, 1))[:, 1]
    return out


# --- 1) Hiệu chuẩn Platt từng model ------------------------------------------
# OOF_P_CAL: cross-fit -> dùng để CHẤM ĐIỂM trung thực trên POOL.
# PLATT_FULL: fit trên toàn bộ OOF -> tham số sẽ áp lên TEST (TEST tách biệt hoàn toàn).
OOF_P_CAL = {n: crossfit_platt(OOF_P[n], FOLD_ID, VALID) for n in CANDIDATES}
PLATT_FULL = {}
for n in CANDIDATES:
    _lr = LogisticRegression(C=1e6, solver="lbfgs", max_iter=1000)
    _lr.fit(_logit(OOF_P[n][VALID]).reshape(-1, 1), y[VALID].astype(int))
    PLATT_FULL[n] = _lr

y_oof = y[VALID].astype(int)
fold_valid = FOLD_ID[VALID]
M_OOF_RAW = np.column_stack([OOF_P[n][VALID] for n in CANDIDATES])
M_OOF_CAL = np.column_stack([OOF_P_CAL[n][VALID] for n in CANDIDATES])
_folds_seen = np.unique(fold_valid)
print(f"Ma trận OOF: {M_OOF_RAW.shape[0]:,} bản ghi × {M_OOF_RAW.shape[1]} model "
      f"| {y_oof.sum()} ca dương | {len(_folds_seen)} fold")


# --- 2) Các cách tổ hợp ------------------------------------------------------
def comb_mean(M, w=None):
    w = np.ones(M.shape[1]) / M.shape[1] if w is None else np.asarray(w, dtype=float)
    return M @ (w / w.sum())


def comb_logit_mean(M, w=None):
    """Trung bình trong không gian logit rồi đưa ngược về xác suất. Ít bị một model
    'tự tin sai' kéo lệch hơn trung bình xác suất thường."""
    w = np.ones(M.shape[1]) / M.shape[1] if w is None else np.asarray(w, dtype=float)
    return 1.0 / (1.0 + np.exp(-(_logit(M) @ (w / w.sum()))))


def greedy_weights(M, yv, n_iter=GREEDY_ITERS):
    """Caruana ensemble selection CÓ HOÀN LẠI: mỗi vòng thêm model làm AUPRC tăng nhiều nhất,
    một model được chọn nhiều lần = trọng số cao. Tự động loại model kém mà không cần tay."""
    n_models = M.shape[1]
    base = np.array([average_precision_score(yv, M[:, j]) for j in range(n_models)])
    j0 = int(np.argmax(base))
    counts = np.zeros(n_models)
    counts[j0] = 1.0
    cur = M[:, j0].copy()
    best_counts, best_score = counts.copy(), float(base[j0])
    for _ in range(n_iter):
        cand = [average_precision_score(yv, (cur + M[:, j]) / (counts.sum() + 1))
                for j in range(n_models)]
        j = int(np.argmax(cand))
        cur = cur + M[:, j]
        counts[j] += 1
        if cand[j] > best_score:
            best_score, best_counts = float(cand[j]), counts.copy()
    return best_counts / best_counts.sum()


def topk_weights(M, yv):
    """Trung bình đều K model tốt nhất theo AUPRC OOF; K cũng được chọn trên OOF."""
    scores = np.array([average_precision_score(yv, M[:, j]) for j in range(M.shape[1])])
    order = np.argsort(scores)[::-1]
    best_k, best_s = 1, -np.inf
    for k in range(1, len(order) + 1):
        s = average_precision_score(yv, M[:, order[:k]].mean(axis=1))
        if s > best_s:
            best_s, best_k = s, k
    w = np.zeros(M.shape[1])
    w[order[:best_k]] = 1.0 / best_k
    return w


def stack_fit(M, yv):
    lr = LogisticRegression(C=STACK_C, solver="lbfgs", max_iter=2000)
    lr.fit(_logit(M), yv.astype(int))
    return lr


def stack_apply(lr, M):
    return lr.predict_proba(_logit(M))[:, 1]


def crossfit_combiner(fit_fn, apply_fn, M):
    """Ước lượng TRUNG THỰC cho một cách tổ hợp CÓ THAM SỐ: fold k được dự đoán bằng tham số
    fit trên các fold khác. Không có bước này, greedy/stacking sẽ tự chấm điểm cho chính nó
    trên dữ liệu nó vừa được fit -> luôn thắng một cách giả tạo."""
    if len(_folds_seen) < 2:                      # debug 1 fold: không cross-fit được
        state = fit_fn(M, y_oof)
        return apply_fn(state, M)
    out = np.full(M.shape[0], np.nan)
    for k in _folds_seen:
        fm, am = fold_valid != k, fold_valid == k
        out[am] = apply_fn(fit_fn(M[fm], y_oof[fm]), M[am])
    return out


# --- 3) Vườn ensemble: điểm OOF trung thực + hàm áp lên TEST ------------------
# ENS_APPLY[tên] = (loại đầu vào "raw"/"cal", hàm(M) -> xác suất) với tham số đã fit trên
# TOÀN BỘ OOF -- đây là thứ sẽ áp nguyên xi lên TEST.
ENS_OOF, ENS_APPLY, ENS_INFO = {}, {}, {}

ENS_OOF["Avg-Raw"] = comb_mean(M_OOF_RAW)
ENS_APPLY["Avg-Raw"] = ("raw", lambda M: comb_mean(M))
ENS_INFO["Avg-Raw"] = "trung bình xác suất thô (cách v2 dùng — mốc đối chiếu)"

ENS_OOF["Avg-Cal"] = comb_mean(M_OOF_CAL)
ENS_APPLY["Avg-Cal"] = ("cal", lambda M: comb_mean(M))
ENS_INFO["Avg-Cal"] = "trung bình xác suất sau hiệu chuẩn Platt"

ENS_OOF["LogitAvg-Cal"] = comb_logit_mean(M_OOF_CAL)
ENS_APPLY["LogitAvg-Cal"] = ("cal", lambda M: comb_logit_mean(M))
ENS_INFO["LogitAvg-Cal"] = "trung bình trong không gian logit sau hiệu chuẩn"

_w_greedy = greedy_weights(M_OOF_CAL, y_oof)
ENS_OOF["Greedy-Cal"] = crossfit_combiner(lambda Mf, yf: greedy_weights(Mf, yf),
                                          lambda w, Ma: comb_mean(Ma, w), M_OOF_CAL)
ENS_APPLY["Greedy-Cal"] = ("cal", lambda M, w=_w_greedy: comb_mean(M, w))
ENS_INFO["Greedy-Cal"] = "Caruana greedy selection tối ưu AUPRC trên OOF"

_w_topk = topk_weights(M_OOF_CAL, y_oof)
ENS_OOF["TopK-Cal"] = crossfit_combiner(lambda Mf, yf: topk_weights(Mf, yf),
                                        lambda w, Ma: comb_mean(Ma, w), M_OOF_CAL)
ENS_APPLY["TopK-Cal"] = ("cal", lambda M, w=_w_topk: comb_mean(M, w))
ENS_INFO["TopK-Cal"] = f"trung bình đều {int((_w_topk > 0).sum())} model tốt nhất theo AUPRC OOF"

_stack_full = stack_fit(M_OOF_CAL, y_oof)
ENS_OOF["Stack-LR"] = crossfit_combiner(stack_fit, stack_apply, M_OOF_CAL)
ENS_APPLY["Stack-LR"] = ("cal", lambda M, lr=_stack_full: stack_apply(lr, M))
ENS_INFO["Stack-LR"] = "hồi quy logistic trên logit của các model (stacking)"

ENSEMBLE_NAMES = list(ENS_OOF)
for _nm, _v in ENS_OOF.items():
    assert np.isfinite(_v).all(), f"{_nm}: vector OOF còn giá trị không hữu hạn"

# --- Trọng số các phương án có tham số ---------------------------------------
_w_df = pd.DataFrame({"Greedy-Cal": _w_greedy, "TopK-Cal": _w_topk,
                      "Stack-LR (hệ số)": _stack_full.coef_.ravel()}, index=CANDIDATES)
print("\nTrọng số mà từng cách tổ hợp gán cho mỗi model (fit trên TOÀN BỘ OOF):")
display(_w_df.style.format("{:+.4f}").background_gradient(cmap="Blues",
                                                          subset=["Greedy-Cal", "TopK-Cal"]))
_dropped = [n for n, w in zip(CANDIDATES, _w_greedy) if w == 0]
print(f"Greedy loại hẳn {len(_dropped)}/{len(CANDIDATES)} model: {', '.join(_dropped) or '(không loại ai)'}")


# --- 4) Bảng chấm điểm trên OOF: model đơn + mọi ensemble --------------------
def ece_of(y_true, y_prob, n_bins=ECE_BINS):
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_prob = np.asarray(y_prob, dtype=float).ravel()
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(y_prob, edges[1:-1]), 0, n_bins - 1)
    ece = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.sum():
            ece += m.sum() / len(y_prob) * abs(float(y_true[m].mean()) - float(y_prob[m].mean()))
    return float(ece)


_rows = []
for n in CANDIDATES:
    m = compute_metrics(y_oof, M_OOF_CAL[:, CANDIDATES.index(n)])
    _rows.append({"Tên": n, "Loại": "model đơn", "AUROC": m["auroc"], "AUPRC": m["auprc"],
                  "Brier": m["brier"], "ECE": ece_of(y_oof, M_OOF_CAL[:, CANDIDATES.index(n)])})
for n in ENSEMBLE_NAMES:
    m = compute_metrics(y_oof, ENS_OOF[n])
    _rows.append({"Tên": n, "Loại": "ensemble", "AUROC": m["auroc"], "AUPRC": m["auprc"],
                  "Brier": m["brier"], "ECE": ece_of(y_oof, ENS_OOF[n])})
OOF_SCORE_DF = pd.DataFrame(_rows).set_index("Tên").sort_values("AUPRC", ascending=False)
print(f"\nChấm điểm trên POOL/OOF ({len(y_oof):,} bản ghi, {y_oof.sum()} dương) — "
      f"ensemble có tham số đã được cross-fit nên điểm là trung thực:")
display(style_df(OOF_SCORE_DF, ["AUROC", "AUPRC", "Brier", "ECE"]))


# --- 5) So sánh cặp có khoảng tin cậy, bootstrap THEO BỆNH NHÂN --------------
_pat_oof = df.loc[VALID, COL_PATIENT].astype(str).to_numpy()


def _patient_boot_index(patients, n_boot, seed):
    """Sinh sẵn chỉ số bootstrap theo bệnh nhân (vector hoá, tránh vòng lặp Python lồng nhau)."""
    order = np.argsort(patients, kind="stable")
    pat_sorted = patients[order]
    uniq = np.unique(pat_sorted)
    starts = np.searchsorted(pat_sorted, uniq, side="left")
    counts = np.searchsorted(pat_sorted, uniq, side="right") - starts
    rng = np.random.default_rng(seed)
    for _ in range(n_boot):
        pick = rng.integers(0, len(uniq), size=len(uniq))
        lens = counts[pick]
        total = int(lens.sum())
        offs = np.arange(total) - np.repeat(np.cumsum(lens) - lens, lens)
        yield order[np.repeat(starts[pick], lens) + offs]


def paired_auprc_ci(y_true, p_a, p_b, patients, n_boot=N_BOOTSTRAP, seed=SEED):
    """CI 95% của AUPRC(a) - AUPRC(b), lấy mẫu bootstrap CÙNG một tập bệnh nhân cho cả hai."""
    diffs = []
    for idx in _patient_boot_index(patients, n_boot, seed):
        yb = y_true[idx]
        if yb.sum() == 0 or yb.sum() == len(yb):
            continue
        diffs.append(average_precision_score(yb, p_a[idx]) - average_precision_score(yb, p_b[idx]))
    d = np.asarray(diffs)
    return float(np.mean(d)), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))


_base = "Avg-Raw"
print(f"\nChênh lệch AUPRC so với {_base} (cách v2 dùng) — bootstrap {N_BOOTSTRAP:,} lần "
      f"theo {len(np.unique(_pat_oof)):,} bệnh nhân của POOL:")
_cmp = []
for n in ENSEMBLE_NAMES:
    if n == _base:
        continue
    d, lo, hi = paired_auprc_ci(y_oof, ENS_OOF[n], ENS_OOF[_base], _pat_oof)
    _cmp.append({"Ensemble": n, "ΔAUPRC": d, "CI thấp": lo, "CI cao": hi,
                 "Có ý nghĩa": "có" if lo > 0 else ("kém hơn" if hi < 0 else "không")})
ENS_COMPARE_DF = pd.DataFrame(_cmp).set_index("Ensemble").sort_values("ΔAUPRC", ascending=False)
display(ENS_COMPARE_DF.style.format({"ΔAUPRC": "{:+.4f}", "CI thấp": "{:+.4f}", "CI cao": "{:+.4f}"}))


# --- 6) KHOÁ quán quân (chọn thuần trên OOF) ---------------------------------
_ens_only = OOF_SCORE_DF[OOF_SCORE_DF["Loại"] == "ensemble"]
PRIMARY_ENSEMBLE = str(_ens_only["AUPRC"].idxmax())
print("\n" + "=" * 86)
print(f"ENSEMBLE CHÍNH ĐƯỢC KHOÁ: {PRIMARY_ENSEMBLE}  ({ENS_INFO[PRIMARY_ENSEMBLE]})")
print(f"  OOF AUPRC {_ens_only.loc[PRIMARY_ENSEMBLE, 'AUPRC']:.4f} | "
      f"AUROC {_ens_only.loc[PRIMARY_ENSEMBLE, 'AUROC']:.4f} | "
      f"ECE {_ens_only.loc[PRIMARY_ENSEMBLE, 'ECE']:.4f}")
print(f"  (mốc Avg-Raw của v2: AUPRC {_ens_only.loc[_base, 'AUPRC']:.4f} -> "
      f"chênh {_ens_only.loc[PRIMARY_ENSEMBLE, 'AUPRC'] - _ens_only.loc[_base, 'AUPRC']:+.4f})")
print("Chọn xong TRƯỚC khi chạm vào TEST. Từ mục 17 trở đi chỉ áp dụng, không chọn lại nữa.")
print("=" * 86)

# Đưa ensemble vào OOF_P để các mục sau dùng chung một giao diện với model đơn
for n in ENSEMBLE_NAMES:
    _full = np.full(len(df), NAN)
    _full[VALID] = ENS_OOF[n]
    OOF_P[n] = _full

## 17. Huấn luyện model FINAL trên toàn bộ POOL

Mỗi kiến trúc được huấn luyện lại **một lần duy nhất** trên 100% POOL (không chỉ 4/5 như lúc
K-Fold) — đây là model dùng để dự đoán trên TEST và dùng nếu triển khai thật. Số epoch lấy
trung vị `best_epoch` quan sát được qua 5 fold ở bước 13 (không có tập val riêng để early-stop
ở đây vì đã dùng hết POOL cho train).

In [ ]:
pool_loader = DataLoader(ECGDataset(POOL_IDX, y[POOL_IDX], mean_pool, std_pool, augment=True),
                         batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                         pin_memory=GPU_AVAILABLE, **_extra)

# --- Kiểm tra độ ổn định best_epoch trước khi chọn epoch cho FINAL ---
# Model FINAL train trên 100% POOL, không có validation riêng để tự early-stop -- số epoch
# dùng phải suy từ trung vị best_epoch của K-Fold. Nếu con số này dao động mạnh giữa các fold,
# trung vị kém tin cậy hơn và cần lưu ý khi diễn giải kết quả FINAL.
print("Độ ổn định best_epoch qua các fold (trước khi chọn epoch cho model FINAL):")
_stab_rows = []
for name in CANDIDATES:
    eps = [RUN_INFO[f"{name}|{k}"]["best_epoch"] for k in range(N_FOLDS_RUN) if f"{name}|{k}" in RUN_INFO]
    if not eps:
        continue
    _stab_rows.append({"Model": name, "min": min(eps), "median": float(np.median(eps)),
                       "max": max(eps), "std": float(np.std(eps))})
_stab_df = pd.DataFrame(_stab_rows).set_index("Model")
display(_stab_df.style.format({"median": "{:.1f}", "std": "{:.2f}"}))
_unstable = _stab_df[_stab_df["std"] > 0.5 * _stab_df["median"].clip(lower=1)]
if len(_unstable):
    print(f"\n!! CẢNH BÁO: {len(_unstable)} model có best_epoch dao động mạnh giữa các fold "
          f"(std > 50% median) -> số epoch dùng cho FINAL kém tin cậy hơn: "
          f"{', '.join(_unstable.index)}")
else:
    print("\nbest_epoch tương đối ổn định giữa các fold cho mọi model -> dùng trung vị hợp lý.")
print("(Ở v3, predictor chính là BAGGING nên độ tin cậy của con số này ảnh hưởng ít hơn v2 --"
      "\n model FINAL chỉ còn là hàng đối chiếu.)")

FINAL_EPOCHS = {}
for name in CANDIDATES:
    eps = [RUN_INFO[f"{name}|{k}"]["best_epoch"] for k in range(N_FOLDS_RUN) if f"{name}|{k}" in RUN_INFO]
    FINAL_EPOCHS[name] = max(1, int(round(np.median(eps)))) if eps else EPOCHS
print("\nSố epoch dùng để train FINAL (trung vị best_epoch qua các fold):")
for name in CANDIDATES:
    print(f"  {name:<16} {FINAL_EPOCHS[name]} epoch")

n_pos_pool = int(y[POOL_IDX].sum())
alpha_pos_pool = 1.0 - n_pos_pool / len(POOL_IDX)


def train_final(name, cls, n_epochs, tag):
    ckpt_path = MODEL_DIR / f'{name.replace("+", "_")}_FINAL.pt'
    fp = dict(target=TARGET_LABEL, n_pool=len(POOL_IDX), model=name, n_epochs=n_epochs, seed=SEED,
              preprocess_version=PREPROCESS_VERSION, focal_gamma=FOCAL_GAMMA,
              label_smooth_eps=LABEL_SMOOTH_EPS, aug_enable=AUG_ENABLE,
              warmup_epochs=WARMUP_EPOCHS, lr=MODEL_LR_OVERRIDE.get(name, LR))
    if ckpt_path.exists():
        ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        if ck.get("fp") == fp:
            model = cls().to(DEVICE)
            model.load_state_dict(ck["model"])
            print(f"  [{tag}] đã có checkpoint FINAL khớp -> bỏ qua train")
            return model

    torch.manual_seed(SEED + 500)
    model = cls().to(DEVICE)
    base_lr = MODEL_LR_OVERRIDE.get(name, LR)
    criterion = FocalLossWithSmoothing(alpha_pos=alpha_pos_pool, gamma=FOCAL_GAMMA,
                                       label_smoothing=LABEL_SMOOTH_EPS)
    optimizer = torch.optim.AdamW(model.parameters(), lr=base_lr, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    t0 = time.time()
    for epoch in range(n_epochs):
        _apply_warmup_lr(optimizer, base_lr, epoch)     # cùng lịch warmup như lúc train fold
        model.train()
        for xb, yb in pool_loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                loss = criterion(model(xb), yb)
            if not torch.isfinite(loss):
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
        if (epoch + 1) % 10 == 0 or epoch == n_epochs - 1:
            print(f"  [{tag}] ep {epoch + 1:>3}/{n_epochs}  {time.time() - t0:.0f}s")
    torch.save({"model": model.state_dict(), "fp": fp}, ckpt_path)
    print(f"  [{tag}] xong {n_epochs} epoch trong {time.time() - t0:.0f}s -- đã lưu {ckpt_path.name}")
    return model


FINAL_MODELS = {}
for name in CANDIDATES:
    tag = f"{name} FINAL({FINAL_EPOCHS[name]}ep)"
    print(f"\n=== {tag} " + "=" * max(1, 50 - len(tag)))
    FINAL_MODELS[name] = train_final(name, MODELS[name], FINAL_EPOCHS[name], tag)
    if GPU_AVAILABLE:
        torch.cuda.empty_cache()

print(f"\nĐã có {len(FINAL_MODELS)} model FINAL (mỗi kiến trúc 1 checkpoint, train trên 100% POOL).")

## 18. Dự đoán trên TEST — bagging (chính) và FINAL (đối chiếu)

Từ đây mới chạm vào tập TEST giữ riêng. Mỗi kiến trúc được dự đoán theo hai cách: trung bình
5 checkpoint fold (**bagging**, predictor chính của v3) và checkpoint FINAL train trên 100%
POOL (cách v2/báo cáo dùng, giữ để đối chiếu).

In [ ]:
y_test = y[TEST_IDX].astype(int)
PAT_TEST = df.loc[TEST_IDX, COL_PATIENT].astype(str).to_numpy()
print(f"TEST: {len(TEST_IDX):,} bản ghi | {len(np.unique(PAT_TEST)):,} bệnh nhân | "
      f"{int(y_test.sum())} dương ({y_test.mean():.2%})")

# =============================================================================
# (a) BAGGING -- trung bình 5 checkpoint fold cho mỗi kiến trúc.
#
# Vì sao đây là predictor chính của v3:
#   1. MẠNH HƠN. Đo trên đúng bộ dữ liệu này (notebook 12): bagging tăng AUPRC của model đơn
#      thêm +0,03..+0,05 so với checkpoint FINAL. Các checkpoint fold đã được train và lưu
#      sẵn ở mục 13 nên mức lợi này KHÔNG tốn thêm một giây GPU nào.
#   2. KHỚP THANG XÁC SUẤT VỚI OOF. Ngưỡng được chốt trên OOF -- vốn cũng do các model train
#      trên ~80% POOL sinh ra. Model FINAL train trên 100% POOL nên tự tin hơn và xác suất
#      lệch thang, khiến ngưỡng chuyển sang TEST bị sai. Đây chính là lỗi làm Sensitivity của
#      v2 tụt còn 0,8894 dù chính sách đặt ra là >= 0,91.
#
# Mỗi fold PHẢI dùng đúng z-score stats của tập train fold đó -- dùng nhầm stats của POOL là
# đưa model vào phân phối đầu vào khác lúc nó được huấn luyện.
# =============================================================================
TEST_P_BAG = {n: None for n in CANDIDATES}
_bag_parts = {n: [] for n in CANDIDATES}

for k in range(N_FOLDS_RUN):
    mean_k, std_k = norm_stats(FOLDS[k][0])
    loader_k = DataLoader(ECGDataset(TEST_IDX, y[TEST_IDX], mean_k, std_k, augment=False),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                          pin_memory=GPU_AVAILABLE, **_extra)
    for name in CANDIDATES:
        ckpt_path = MODEL_DIR / f'{name.replace("+", "_")}_f{k}_best.pt'
        if not ckpt_path.exists():
            print(f"  !! thiếu {ckpt_path.name} -> bỏ qua fold {k} của {name}")
            continue
        model = MODELS[name]().to(DEVICE)
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE,
                                         weights_only=False)["model"])
        _, p = evaluate(model, loader_k)
        _bag_parts[name].append(p)
        del model
        if GPU_AVAILABLE:
            torch.cuda.empty_cache()
    print(f"  fold {k}: đã dự đoán TEST bằng {sum(len(v) > k for v in _bag_parts.values())}"
          f"/{len(CANDIDATES)} kiến trúc")

for name in CANDIDATES:
    assert _bag_parts[name], f"Không có checkpoint fold nào của {name} -- không bagging được"
    TEST_P_BAG[name] = np.mean(_bag_parts[name], axis=0)
print(f"\nBagging xong: mỗi kiến trúc trung bình {len(_bag_parts[CANDIDATES[0]])}"
      f"/{N_FOLDS_RUN} checkpoint fold.")

# =============================================================================
# (b) FINAL -- 1 checkpoint train trên 100% POOL (cách v2/báo cáo dùng), giữ để đối chiếu.
# =============================================================================
TEST_P_FIN = {}
_loader_final = DataLoader(ECGDataset(TEST_IDX, y[TEST_IDX], mean_pool, std_pool, augment=False),
                           batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                           pin_memory=GPU_AVAILABLE, **_extra)
for name in CANDIDATES:
    _, TEST_P_FIN[name] = evaluate(FINAL_MODELS[name], _loader_final)


# =============================================================================
# (c) Dựng toàn bộ ensemble trên TEST bằng ĐÚNG tham số đã khoá ở mục 16
# =============================================================================
def build_test_ensembles(single_probs):
    """Áp mọi cách tổ hợp trong ENS_APPLY lên một bộ dự đoán TEST của các model đơn.
    Hiệu chuẩn Platt dùng PLATT_FULL (fit trên toàn bộ OOF của POOL) -- không nhìn TEST."""
    M_raw = np.column_stack([single_probs[n] for n in CANDIDATES])
    M_cal = np.column_stack([
        PLATT_FULL[n].predict_proba(_logit(single_probs[n]).reshape(-1, 1))[:, 1]
        for n in CANDIDATES])
    out = dict(single_probs)
    for nm, (kind, fn) in ENS_APPLY.items():
        out[nm] = fn(M_cal if kind == "cal" else M_raw)
    return out


TEST_P_BAG = build_test_ensembles(TEST_P_BAG)
TEST_P_FIN = build_test_ensembles(TEST_P_FIN)
TEST_P = TEST_P_BAG if PRIMARY_PREDICTOR == "bagging" else TEST_P_FIN
ALL_NAMES = CANDIDATES + ENSEMBLE_NAMES

# --- So sánh hai predictor trên TEST (chỉ AUROC/AUPRC, không phụ thuộc ngưỡng) ---
_rows = []
for name in ALL_NAMES:
    mb = compute_metrics(y_test, TEST_P_BAG[name])
    mf = compute_metrics(y_test, TEST_P_FIN[name])
    _rows.append({"Tên": name, "Loại": "ensemble" if name in ENSEMBLE_NAMES else "model đơn",
                  "AUROC bagging": mb["auroc"], "AUROC final": mf["auroc"],
                  "AUPRC bagging": mb["auprc"], "AUPRC final": mf["auprc"],
                  "ΔAUPRC": mb["auprc"] - mf["auprc"]})
BAG_VS_FINAL_DF = pd.DataFrame(_rows).set_index("Tên")
print("\n" + "=" * 90)
print("TEST — BAGGING (nhiều checkpoint fold) so với FINAL (1 checkpoint trên 100% POOL)")
print("=" * 90)
display(style_df(BAG_VS_FINAL_DF, ["AUROC bagging", "AUROC final", "AUPRC bagging",
                                   "AUPRC final", "ΔAUPRC"]))
_win = int((BAG_VS_FINAL_DF["ΔAUPRC"] > 0).sum())
print(f"Bagging tốt hơn ở {_win}/{len(BAG_VS_FINAL_DF)} hàng "
      f"(trung bình ΔAUPRC {BAG_VS_FINAL_DF['ΔAUPRC'].mean():+.4f}).")
print(f"Predictor dùng cho mọi bảng phía dưới: {PRIMARY_PREDICTOR.upper()}")

## 19. Chốt ngưỡng có biên an toàn → bảng kết quả TEST → so với báo cáo

In [ ]:
# =============================================================================
# Chốt ngưỡng trên POOL, CÓ BIÊN AN TOÀN.
#
# Vấn đề mà mục này sửa: ngưỡng chốt để đạt Sensitivity = S trên POOL gần như KHÔNG BAO GIỜ
# cho đúng S trên TEST. TEST chỉ có ~217 ca dương, nên Sensitivity ở đó dao động thuần do cỡ
# mẫu khoảng ±2 điểm %. Ở v2 sai số này rơi vào chiều xấu: chốt 91% trên POOL nhưng chỉ đạt
# 88,94% trên TEST -- tức chính sách lâm sàng đã đặt ra bị vi phạm.
#
# Cách sửa (thuần POOL, không hề nhìn TEST): tại đúng ngưỡng vừa chốt, đo Sensitivity đạt được
# trong TỪNG fold rồi lấy độ lệch chuẩn giữa các fold. Mỗi fold có ~245 ca dương -- cỡ tương
# đương TEST -- nên con số này chính là mức dao động ta sẽ gặp. Nhắm cao hơn mục tiêu z lần độ
# lệch đó thì xác suất đạt mục tiêu thật trên TEST là ~90% (z = 1,28).
# =============================================================================


def fold_sensitivity_sd(p_oof_valid, thr):
    """Độ lệch chuẩn của Sensitivity giữa các fold tại CÙNG một ngưỡng."""
    sens = []
    for k in _folds_seen:
        m = fold_valid == k
        yk, pk = y_oof[m], p_oof_valid[m]
        if yk.sum() == 0:
            continue
        sens.append(float((pk[yk == 1] >= thr).mean()))
    return float(np.std(sens, ddof=1)) if len(sens) > 1 else 0.0


def threshold_with_margin(p_oof_valid, target, z=SENS_MARGIN_Z, use_margin=USE_SENS_MARGIN):
    """Trả về (ngưỡng, mục_tiêu_đã_nâng, sd_giữa_các_fold, có_chạm_trần).

    CHẶN TRÊN là bắt buộc: nếu độ lệch giữa các fold lớn bất thường, target + z*sd có thể tiến
    sát 1,0 và ngưỡng tụt xuống mức dự đoán MỌI ca là dương (Specificity = 0). Đó là hỏng âm
    thầm -- bảng vẫn in ra bình thường. Chặn ở SENS_MARGIN_MAX_TARGET và cảnh báo tường minh."""
    t_plain = threshold_at_sensitivity(y_oof, p_oof_valid, target)
    sd = fold_sensitivity_sd(p_oof_valid, t_plain)
    if not use_margin:
        return t_plain, target, sd, False
    raw_adj = target + z * sd
    target_adj = float(min(raw_adj, SENS_MARGIN_MAX_TARGET))
    capped = raw_adj > SENS_MARGIN_MAX_TARGET + 1e-12
    return threshold_at_sensitivity(y_oof, p_oof_valid, target_adj), target_adj, sd, capped


POOL_THRESHOLDS, THR_INFO = {}, {}
_rows, _capped_any = [], []
for name in ALL_NAMES:
    p_ = OOF_P[name][VALID]
    thr, tgt_adj, sd, capped = threshold_with_margin(p_, TARGET_SENSITIVITY)
    POOL_THRESHOLDS[name] = thr
    THR_INFO[name] = {"target_adj": tgt_adj, "fold_sd": sd, "capped": capped}
    if capped:
        _capped_any.append(name)
    m = compute_metrics(y_oof, p_, thr)
    _rows.append({"Tên": name, "Loại": "ensemble" if name in ENSEMBLE_NAMES else "model đơn",
                  "Ngưỡng": thr, "Mục tiêu đã nâng": tgt_adj, "SD giữa fold": sd,
                  "Sens trên POOL": m["sensitivity"], "Spec trên POOL": m["specificity"],
                  "AUPRC POOL": m["auprc"], "Chạm trần biên": "có" if capped else ""})
POOL_THR_DF = pd.DataFrame(_rows).set_index("Tên")
print(f"Chốt ngưỡng trên POOL/OOF ({len(y_oof):,} bản ghi, {y_oof.sum()} dương) — "
      f"mục tiêu Sensitivity >= {TARGET_SENSITIVITY:.0%}"
      + (f", biên an toàn z={SENS_MARGIN_Z}" if USE_SENS_MARGIN else ", KHÔNG biên"))
display(style_df(POOL_THR_DF, ["Ngưỡng", "Mục tiêu đã nâng", "SD giữa fold",
                               "Sens trên POOL", "Spec trên POOL", "AUPRC POOL"]))
_pi = THR_INFO[PRIMARY_ENSEMBLE]
print(f"{PRIMARY_ENSEMBLE}: SD Sensitivity giữa các fold = {_pi['fold_sd']:.4f} -> "
      f"nhắm {_pi['target_adj']:.4f} trên POOL để kỳ vọng đạt >= {TARGET_SENSITIVITY:.0%} trên TEST.")
if _capped_any:
    print(f"\n!! CẢNH BÁO: {len(_capped_any)} model bị chặn ở trần biên an toàn "
          f"{SENS_MARGIN_MAX_TARGET:.0%}: {', '.join(_capped_any)}")
    print("   Nghĩa là Sensitivity của chúng dao động rất mạnh giữa các fold. Ngưỡng đã bị")
    print("   chặn để không tụt xuống mức dự đoán mọi ca là dương (Specificity = 0).")
    print("   Nếu mô hình chính nằm trong danh sách này, hãy xem lại độ ổn định huấn luyện")
    print("   trước khi dùng con số của nó cho báo cáo.")


# =============================================================================
# BẢNG KẾT QUẢ CUỐI CÙNG TRÊN TEST
# =============================================================================
_rows = []
for name in ALL_NAMES:
    t = POOL_THRESHOLDS[name]
    m = compute_metrics(y_test, TEST_P[name], t)
    _rows.append({"Tên": name, "Loại": "ensemble" if name in ENSEMBLE_NAMES else "model đơn",
                  "Ngưỡng (từ POOL)": t, "AUROC": m["auroc"], "AUPRC": m["auprc"],
                  "Sensitivity": m["sensitivity"], "Specificity": m["specificity"],
                  "PPV": m["ppv"], "NPV": m["npv"], "F1": m["f1"],
                  "TP": m["tp"], "FN": m["fn"], "FP": m["fp"], "TN": m["tn"]})
FINAL_TEST_DF = pd.DataFrame(_rows).set_index("Tên").sort_values("AUPRC", ascending=False)

print("\n" + "=" * 96)
print(f"KẾT QUẢ TEST GIỮ RIÊNG ({len(TEST_IDX):,} bản ghi, {int(y_test.sum())} dương) — "
      f"predictor {PRIMARY_PREDICTOR.upper()}, ngưỡng chốt từ POOL")
print("=" * 96)
display(style_df(FINAL_TEST_DF, ["Ngưỡng (từ POOL)", "AUROC", "AUPRC", "Sensitivity",
                                 "Specificity", "PPV", "NPV", "F1"]))

CHAMPION = PRIMARY_ENSEMBLE       # đã khoá trên OOF ở mục 16, KHÔNG chọn lại theo điểm TEST
_champ = FINAL_TEST_DF.loc[CHAMPION]
print(f"\nMô hình chính (đã khoá trên POOL từ mục 16): {CHAMPION} — {ENS_INFO[CHAMPION]}")
_sens_ok = _champ["Sensitivity"] >= TARGET_SENSITIVITY
print(f"  Chính sách Sensitivity >= {TARGET_SENSITIVITY:.0%}: đạt {_champ['Sensitivity']:.4f} -> "
      f"{'ĐẠT' if _sens_ok else 'CHƯA ĐẠT'}")
_best_auprc_test = FINAL_TEST_DF["AUPRC"].idxmax()
if _best_auprc_test != CHAMPION:
    print(f"  (Ghi chú trung thực: trên TEST, {_best_auprc_test} có AUPRC cao hơn "
          f"({FINAL_TEST_DF.loc[_best_auprc_test, 'AUPRC']:.4f} so với {_champ['AUPRC']:.4f}). "
          f"KHÔNG đổi mô hình chính theo con số này —\n   chọn theo điểm TEST chính là hình thức "
          f"rò rỉ tập test mà toàn bộ thiết kế này đang tránh.)")


# =============================================================================
# SO SÁNH VỚI SỐ LIỆU ĐANG CÓ TRONG BÁO CÁO (mục 2.2.1)
# =============================================================================
_higher_better = {"AUROC": True, "AUPRC": True, "Sensitivity": True, "Specificity": True,
                  "PPV": True, "NPV": True, "F1": True, "TP": True, "TN": True,
                  "FN": False, "FP": False}
_rows = []
for k, better_up in _higher_better.items():
    new, old = float(_champ[k]), float(REPORT_BASELINE[k])
    delta = new - old
    if not np.isfinite(delta):
        verdict = "n/a"                       # chỉ số không xác định được (vd. NPV khi TN+FN=0)
    elif delta == 0:
        verdict = "bằng"
    else:
        verdict = "tốt hơn" if (delta > 0) == better_up else "kém hơn"
    _rows.append({"Chỉ số": k, "Báo cáo hiện tại": old, f"v3 ({CHAMPION})": new,
                  "Δ": delta, "Đánh giá": verdict})
VS_REPORT_DF = pd.DataFrame(_rows).set_index("Chỉ số")
print("\n" + "=" * 96)
print("SO SÁNH VỚI BÁO CÁO HIỆN TẠI (Bao_cao_nghien_cuu.docx, mục 2.2.1)")
print(f"  Mốc báo cáo: {REPORT_BASELINE_NOTE}")
print(f"  v3         : {CHAMPION}, predictor {PRIMARY_PREDICTOR}, "
      f"TEST {len(TEST_IDX):,} bản ghi / {int(y_test.sum())} dương")
print("=" * 96)
_COUNT_ROWS = ("TP", "FN", "FP", "TN")


def _fmt_col(col, signed=False):
    """TP/FN/FP/TN là số đếm -> in nguyên; các chỉ số còn lại là tỷ lệ -> 4 chữ số thập phân."""
    out = []
    for idx, v in zip(VS_REPORT_DF.index, col):
        if not np.isfinite(v):
            out.append("n/a")
        elif idx in _COUNT_ROWS:
            out.append(f"{int(round(v)):+,d}" if signed else f"{int(round(v)):,d}")
        else:
            out.append(f"{v:+.4f}" if signed else f"{v:.4f}")
    return out


_show = VS_REPORT_DF.copy()
_show["Báo cáo hiện tại"] = _fmt_col(VS_REPORT_DF["Báo cáo hiện tại"])
_show[f"v3 ({CHAMPION})"] = _fmt_col(VS_REPORT_DF[f"v3 ({CHAMPION})"])
_show["Δ"] = _fmt_col(VS_REPORT_DF["Δ"], signed=True)
display(_show)
_n_better = int((VS_REPORT_DF["Đánh giá"] == "tốt hơn").sum())
print(f"Tốt hơn báo cáo ở {_n_better}/{len(VS_REPORT_DF)} chỉ số.")
print("LƯU Ý: hai lần chạy không dùng đúng một danh sách bản ghi TEST (báo cáo: 2.681; v3 loại"
      "\n2 bản ghi lỗi đọc nên còn ~2.677) và số liệu báo cáo không kèm khoảng tin cậy, nên đây"
      "\nlà so sánh điểm-với-điểm, không phải kiểm định thống kê.")

## 20. Khoảng tin cậy 95% — bootstrap lấy mẫu theo bệnh nhân

Một bệnh nhân có thể có nhiều ECG trong TEST; coi chúng là mẫu độc lập sẽ cho khoảng tin cậy
hẹp giả tạo. Lấy mẫu theo bệnh nhân là cách Phần I của báo cáo đã dùng — giữ nhất quán để hai
phần so sánh được với nhau.

In [ ]:
# =============================================================================
# Khoảng tin cậy 95% cho mô hình chính, bootstrap LẤY MẪU THEO BỆNH NHÂN.
# Một bệnh nhân có thể có nhiều ECG trong TEST; coi chúng là các mẫu độc lập sẽ cho CI hẹp
# giả tạo. Lấy mẫu theo bệnh nhân (giữ trọn bộ ECG của người được chọn) là cách Phần I của
# báo cáo đã dùng -- giữ nhất quán để hai phần so sánh được với nhau.
# =============================================================================
_BOOT_KEYS = ["auroc", "auprc", "sensitivity", "specificity", "ppv", "npv", "f1"]
_BOOT_LABEL = {"auroc": "AUROC", "auprc": "AUPRC", "sensitivity": "Sensitivity",
               "specificity": "Specificity", "ppv": "PPV", "npv": "NPV", "f1": "F1"}


def bootstrap_ci(y_true, p, patients, thr, n_boot=N_BOOTSTRAP, seed=SEED):
    acc = {k: [] for k in _BOOT_KEYS}
    n_used = 0
    for idx in _patient_boot_index(patients, n_boot, seed):
        yb = y_true[idx]
        if yb.sum() == 0 or yb.sum() == len(yb):
            continue
        m = compute_metrics(yb, p[idx], thr)
        for k in _BOOT_KEYS:
            acc[k].append(m[k])
        n_used += 1
    point = compute_metrics(y_true, p, thr)
    rows = []
    for k in _BOOT_KEYS:
        a = np.asarray(acc[k], dtype=float)
        a = a[np.isfinite(a)]
        # Một chỉ số có thể không xác định được trong MỌI lần lấy mẫu (ví dụ NPV khi ngưỡng
        # thấp tới mức không có ca nào được dự đoán âm -> TN+FN = 0). Khi đó trả NaN thay vì
        # để np.percentile vỡ trên mảng rỗng -- không được phép sập ở bước cuối sau nhiều giờ GPU.
        lo, hi = ((float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5)))
                  if len(a) else (float("nan"), float("nan")))
        rows.append({"Chỉ số": _BOOT_LABEL[k], "Điểm": point[k], "CI thấp": lo, "CI cao": hi,
                     "Số lần hợp lệ": len(a)})
    return pd.DataFrame(rows).set_index("Chỉ số"), n_used


print(f"Bootstrap {N_BOOTSTRAP:,} lần theo {len(np.unique(PAT_TEST)):,} bệnh nhân của TEST "
      f"(mô hình {CHAMPION}, ngưỡng {POOL_THRESHOLDS[CHAMPION]:.4f}) ...")
BOOT_DF, _n_used = bootstrap_ci(y_test, TEST_P[CHAMPION], PAT_TEST, POOL_THRESHOLDS[CHAMPION])
BOOT_DF["Báo cáo hiện tại"] = [REPORT_BASELINE[i] for i in BOOT_DF.index]
BOOT_DF["Báo cáo nằm trong CI"] = [
    "n/a" if not (np.isfinite(lo) and np.isfinite(hi))
    else ("trong CI" if lo <= b <= hi else ("dưới CI" if b < lo else "trên CI"))
    for lo, hi, b in zip(BOOT_DF["CI thấp"], BOOT_DF["CI cao"], BOOT_DF["Báo cáo hiện tại"])]
display(BOOT_DF.style.format({"Điểm": "{:.4f}", "CI thấp": "{:.4f}", "CI cao": "{:.4f}",
                              "Báo cáo hiện tại": "{:.4f}"}, na_rep="n/a"))
print(f"({_n_used:,}/{N_BOOTSTRAP:,} lần bootstrap hợp lệ — các lần bốc trúng toàn ca âm bị bỏ.)")
print("\nCột cuối đọc thế này: 'trong CI' nghĩa là chênh lệch giữa v3 và báo cáo nằm gọn trong")
print("dao động do cỡ mẫu TEST, chưa đủ để kết luận bên nào thật sự hơn ở chỉ số đó.")

# --- Chênh lệch giữa mô hình chính và mốc Avg-Raw của v2, có CI (cùng tập TEST) ---
if "Avg-Raw" in TEST_P and CHAMPION != "Avg-Raw":
    d, lo, hi = paired_auprc_ci(y_test, TEST_P[CHAMPION], TEST_P["Avg-Raw"], PAT_TEST)
    print(f"\nΔAUPRC trên TEST, {CHAMPION} so với Avg-Raw (cách tổ hợp của v2): "
          f"{d:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}] -> "
          f"{'có ý nghĩa' if lo > 0 else ('kém hơn' if hi < 0 else 'chưa có ý nghĩa')}")

## 20b. Quét điểm vận hành Sensitivity 91% → 95%

Bảng này để chọn điểm vận hành lâm sàng, **không** phải để chọn model. Mọi ngưỡng vẫn chốt
trên POOL (có biên an toàn) rồi áp lên TEST.

In [ ]:
# =============================================================================
# Quét điểm vận hành: Sensitivity mục tiêu 91% -> 95%.
# Mọi ngưỡng đều chốt trên POOL (có biên an toàn) rồi áp lên TEST -- không tối ưu trên TEST.
# Bảng này để chọn điểm vận hành lâm sàng, không phải để chọn model.
# =============================================================================
_sweep_rows = []
for target in SENS_TARGETS:
    for name in ALL_NAMES:
        p_oof_ = OOF_P[name][VALID]
        thr, tgt_adj, sd, _cap = threshold_with_margin(p_oof_, target)
        m = compute_metrics(y_test, TEST_P[name], thr)
        _sweep_rows.append({"Mục tiêu": f"{target:.0%}", "Tên": name,
                            "Loại": "ensemble" if name in ENSEMBLE_NAMES else "model đơn",
                            "Ngưỡng": thr, "AUROC": m["auroc"], "AUPRC": m["auprc"],
                            "Sensitivity": m["sensitivity"], "Specificity": m["specificity"],
                            "PPV": m["ppv"], "NPV": m["npv"], "F1": m["f1"],
                            "TP": m["tp"], "FN": m["fn"], "FP": m["fp"], "TN": m["tn"],
                            "Đạt mục tiêu": "đạt" if m["sensitivity"] >= target else "chưa"})
SWEEP_DF = pd.DataFrame(_sweep_rows)

_SW_COLS = ["Ngưỡng", "AUROC", "AUPRC", "Sensitivity", "Specificity", "PPV", "NPV", "F1"]
for target in SENS_TARGETS:
    sub = (SWEEP_DF[SWEEP_DF["Mục tiêu"] == f"{target:.0%}"]
           .set_index("Tên").sort_values("AUPRC", ascending=False))
    print("=" * 96)
    print(f"ÉP NGƯỠNG Sensitivity >= {target:.0%} — TEST giữ riêng, {len(TEST_IDX):,} bản ghi, "
          f"predictor {PRIMARY_PREDICTOR.upper()}")
    print("=" * 96)
    display(style_df(sub[_SW_COLS + ["TP", "FN", "FP", "TN", "Đạt mục tiêu"]], _SW_COLS))
    r = sub.loc[CHAMPION]
    print(f"{CHAMPION}: ngưỡng {r['Ngưỡng']:.4f} | Sens {r['Sensitivity']:.4f} "
          f"({r['Đạt mục tiêu']}) | Spec {r['Specificity']:.4f} | PPV {r['PPV']:.4f} | "
          f"F1 {r['F1']:.4f}\n")

_champ_sweep = SWEEP_DF[SWEEP_DF["Tên"] == CHAMPION].set_index("Mục tiêu")
print("Tóm tắt điểm vận hành của mô hình chính — đánh đổi Sensitivity/Specificity:")
display(style_df(_champ_sweep[_SW_COLS + ["TP", "FN", "FP", "TN", "Đạt mục tiêu"]], _SW_COLS))
_n_hit = int((_champ_sweep["Đạt mục tiêu"] == "đạt").sum())
print(f"Đạt đúng mục tiêu Sensitivity ở {_n_hit}/{len(SENS_TARGETS)} mức "
      f"({'biên an toàn đang hoạt động' if USE_SENS_MARGIN else 'chưa bật biên an toàn'}).")

## 21. Confusion matrix, ROC và Precision-Recall trên TEST

In [ ]:
def plot_confusion(ax, m, title):
    cm = np.array([[m["tn"], m["fp"]], [m["fn"], m["tp"]]], dtype=float)
    row_sum = cm.sum(axis=1, keepdims=True)
    cm_pct = np.divide(cm, row_sum, out=np.zeros_like(cm), where=row_sum > 0)
    ax.imshow(cm_pct, cmap="Blues", vmin=0, vmax=1)
    labels = [["TN", "FP"], ["FN", "TP"]]
    for i in range(2):
        for j in range(2):
            color = "white" if cm_pct[i, j] > 0.55 else "black"
            ax.text(j, i, f"{labels[i][j]}\n{int(cm[i, j]):,}\n({cm_pct[i, j]:.1%})",
                    ha="center", va="center", fontsize=9, color=color)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Dự đoán Âm", "Dự đoán Dương"], fontsize=8)
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Âm thật", "Dương thật"], fontsize=8)
    ax.set_title(f"{title}\nSens={m['sensitivity']:.3f}  Spec={m['specificity']:.3f}", fontsize=9)


# --- 1) Confusion matrix của mô hình chính, so kề với số liệu báo cáo ---------
_m_champ = compute_metrics(y_test, TEST_P[CHAMPION], POOL_THRESHOLDS[CHAMPION])
_m_report = {"tn": REPORT_BASELINE["TN"], "fp": REPORT_BASELINE["FP"],
             "fn": REPORT_BASELINE["FN"], "tp": REPORT_BASELINE["TP"],
             "sensitivity": REPORT_BASELINE["Sensitivity"],
             "specificity": REPORT_BASELINE["Specificity"]}
fig, axes = plt.subplots(1, 2, figsize=(8.4, 4.0))
plot_confusion(axes[0], _m_report, "BÁO CÁO HIỆN TẠI\n(Average Ensemble, FINAL)")
plot_confusion(axes[1], _m_champ, f"v3 — {CHAMPION}\n({PRIMARY_PREDICTOR})")
fig.suptitle(f"Confusion matrix trên TEST — STEMI là lớp dương "
             f"(ngưỡng chốt từ POOL, mục tiêu Sensitivity >= {TARGET_SENSITIVITY:.0%})",
             y=1.04, fontsize=11)
plt.tight_layout()
plt.show()

print(f"Báo cáo : TP {REPORT_BASELINE['TP']}  FN {REPORT_BASELINE['FN']}  "
      f"FP {REPORT_BASELINE['FP']}  TN {REPORT_BASELINE['TN']}")
print(f"v3      : TP {_m_champ['tp']}  FN {_m_champ['fn']}  "
      f"FP {_m_champ['fp']}  TN {_m_champ['tn']}")

# --- 2) Toàn bộ model đơn + ensemble -----------------------------------------
_ncols = 4
_nrows = int(np.ceil(len(ALL_NAMES) / _ncols))
fig, axes = plt.subplots(_nrows, _ncols, figsize=(3.1 * _ncols, 3.4 * _nrows))
axes = np.atleast_2d(axes)
for i, name in enumerate(ALL_NAMES):
    ax = axes[i // _ncols, i % _ncols]
    m = compute_metrics(y_test, TEST_P[name], POOL_THRESHOLDS[name])
    plot_confusion(ax, m, name + (" ★" if name == CHAMPION else ""))
for i in range(len(ALL_NAMES), _nrows * _ncols):
    axes[i // _ncols, i % _ncols].axis("off")
fig.suptitle(f"Confusion matrix trên TEST ({len(TEST_IDX):,} bản ghi) — predictor "
             f"{PRIMARY_PREDICTOR.upper()}, ngưỡng Sensitivity >= {TARGET_SENSITIVITY:.0%} "
             f"chốt từ POOL  (★ = mô hình chính)", y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

# --- 3) Đường ROC và PR của mô hình chính ------------------------------------
from sklearn.metrics import precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
_fpr, _tpr, _ = roc_curve(y_test, TEST_P[CHAMPION])
axes[0].plot(_fpr, _tpr, lw=1.8, label=f"{CHAMPION} (AUROC {_m_champ['auroc']:.4f})")
axes[0].plot([0, 1], [0, 1], "--", lw=0.8, color="gray")
axes[0].scatter([1 - _m_champ["specificity"]], [_m_champ["sensitivity"]], zorder=5, s=45,
                color="crimson", label="điểm vận hành đã chốt")
axes[0].scatter([1 - REPORT_BASELINE["Specificity"]], [REPORT_BASELINE["Sensitivity"]],
                zorder=5, s=45, marker="^", color="darkorange", label="báo cáo hiện tại")
axes[0].set_xlabel("1 − Specificity"); axes[0].set_ylabel("Sensitivity")
axes[0].set_title("ROC — TEST"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.25)

_prec, _rec, _ = precision_recall_curve(y_test, TEST_P[CHAMPION])
axes[1].plot(_rec, _prec, lw=1.8, label=f"{CHAMPION} (AUPRC {_m_champ['auprc']:.4f})")
axes[1].axhline(y_test.mean(), ls="--", lw=0.8, color="gray",
                label=f"tỷ lệ dương {y_test.mean():.2%}")
axes[1].scatter([_m_champ["sensitivity"]], [_m_champ["ppv"]], zorder=5, s=45, color="crimson",
                label="điểm vận hành đã chốt")
axes[1].scatter([REPORT_BASELINE["Sensitivity"]], [REPORT_BASELINE["PPV"]], zorder=5, s=45,
                marker="^", color="darkorange", label="báo cáo hiện tại")
axes[1].set_xlabel("Recall (Sensitivity)"); axes[1].set_ylabel("Precision (PPV)")
axes[1].set_title("Precision-Recall — TEST"); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 22. Lưu toàn bộ bảng kết quả và xác suất dự đoán

In [ ]:
RESULTS_DIR = PERSIST_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def _save(obj, stem, index=True):
    path = RESULTS_DIR / f"{stem}_{EXPERIMENT_TAG}.csv"
    obj.to_csv(path, index=index, encoding="utf-8-sig")
    print(f"  {path.name:<52} {len(obj):>5} dòng")
    return path


print("Đã lưu vào", RESULTS_DIR)
_save(FINAL_TEST_DF, "stemi_test_metrics")             # bảng kết quả TEST chính
_save(VS_REPORT_DF, "stemi_vs_report")                 # chênh lệch với báo cáo
_save(BOOT_DF, "stemi_test_bootstrap_ci")              # CI 95% theo bệnh nhân
_save(SWEEP_DF, "stemi_sens_sweep", index=False)       # quét điểm vận hành 91-95%
_save(OOF_SCORE_DF, "stemi_oof_model_ranking")         # bảng chấm điểm OOF (cơ sở chọn model)
_save(ENS_COMPARE_DF, "stemi_ensemble_compare_oof")    # ΔAUPRC các ensemble, có CI
_save(POOL_THR_DF, "stemi_pool_thresholds")            # ngưỡng + biên an toàn
_save(BAG_VS_FINAL_DF, "stemi_bagging_vs_final")       # bagging so với FINAL
_save(FOLD_METRICS, "stemi_fold_metrics", index=False)  # kết quả từng fold
_save(focal_diag_df, "focal_weight_diagnostic")
if len(QC_FLATLINE_REPORT):
    _save(QC_FLATLINE_REPORT, "qc_flatline", index=False)

# Lưu luôn xác suất dự đoán trên TEST -- cần cho mọi phân tích lỗi / vẽ lại về sau mà không
# phải chạy lại GPU, và cho phép kiểm định cặp với các lần chạy khác trên cùng tập bản ghi.
_pred_path = RESULTS_DIR / f"stemi_test_predictions_{EXPERIMENT_TAG}.npz"
np.savez_compressed(
    _pred_path, record_stem=df.loc[TEST_IDX, "record_stem"].astype(str).to_numpy(),
    patient=PAT_TEST, y_true=y_test,
    **{f"bag_{n}": TEST_P_BAG[n] for n in ALL_NAMES},
    **{f"fin_{n}": TEST_P_FIN[n] for n in ALL_NAMES})
print(f"  {_pred_path.name:<52} xác suất TEST của {len(ALL_NAMES)} model × 2 predictor")

print("\n" + "=" * 96)
print("TÓM TẮT")
print("=" * 96)
print(f"Mô hình chính   : {CHAMPION} ({ENS_INFO[CHAMPION]}), predictor {PRIMARY_PREDICTOR}")
print(f"Chọn dựa trên   : AUPRC OOF của POOL ({int(VALID.sum()):,} bản ghi) — khoá trước khi mở TEST")
print(f"Ngưỡng vận hành : {POOL_THRESHOLDS[CHAMPION]:.4f} "
      f"(mục tiêu Sens {TARGET_SENSITIVITY:.0%}, nâng lên {THR_INFO[CHAMPION]['target_adj']:.4f} "
      f"để bù dao động cỡ mẫu)")
print(f"TEST            : {len(TEST_IDX):,} bản ghi / {len(np.unique(PAT_TEST)):,} bệnh nhân / "
      f"{int(y_test.sum())} ca STEMI")
print()
for k in ["AUROC", "AUPRC", "Sensitivity", "Specificity", "PPV", "NPV", "F1"]:
    row = VS_REPORT_DF.loc[k]
    print(f"  {k:<12} {row[f'v3 ({CHAMPION})']:.4f}   "
          f"(báo cáo {row['Báo cáo hiện tại']:.4f}, {row['Δ']:+.4f} — {row['Đánh giá']})")
print(f"  {'TP/FN/FP/TN':<12} {_m_champ['tp']}/{_m_champ['fn']}/{_m_champ['fp']}/{_m_champ['tn']}"
      f"   (báo cáo {REPORT_BASELINE['TP']}/{REPORT_BASELINE['FN']}/"
      f"{REPORT_BASELINE['FP']}/{REPORT_BASELINE['TN']})")
print("\nMọi con số trên đến từ MỘT lần đánh giá duy nhất trên tập TEST giữ riêng, sau khi model,")
print("cách tổ hợp và ngưỡng đều đã được khoá trên POOL. Không có bước nào chọn theo điểm TEST.")